# Muse-Glimmer-30B — HF-Only Agent Server Qualification on Kaggle T4x2 / Qualification

This notebook qualifies the repository's private agent-serving stack on Kaggle GPU T4x2. Model weights are downloaded from pinned Hugging Face revisions; they are not required as Kaggle Model attachments.

## Before you run / Trước khi chạy

### English

In Kaggle Notebook settings:

1. Select **GPU T4 x2** and enable Internet.
2. Do **not** attach the target GGUF, DFlash2 draft or projector as Kaggle Models. The project downloads all three from Hugging Face using immutable revisions, 32 parallel file workers, 32 Xet range requests, retries/resume and SHA-256 checks.
3. The pinned llama.cpp CUDA runtime is built from the configured immutable Git commit if it is not already cached.
4. Set MUSE_API_TOKEN as a Kaggle Secret if desired. If omitted, this qualification notebook generates a temporary token and removes it during cleanup.
5. The notebook uses Cloudflare Quick Tunnel for its optional transport qualification. For a stable Named Tunnel, configure the CLI with EXPOSURE_MODE=named, TUNNEL_TOKEN and EXPOSURE_PUBLIC_URL; see the bilingual agent-server guide.

### Tiếng Việt

Trong cài đặt Kaggle Notebook:

1. Chọn **GPU T4 x2** và bật Internet.
2. **Không cần attach** target GGUF, draft DFlash2 hoặc projector dưới dạng Kaggle Model. Dự án tải cả ba từ Hugging Face theo revision bất biến, 32 worker file, 32 Xet range request, retry/resume và kiểm tra SHA-256.
3. Nếu runtime CUDA của llama.cpp chưa có trong cache, dự án build từ Git commit bất biến đã cấu hình.
4. Có thể đặt MUSE_API_TOKEN bằng Kaggle Secret. Nếu thiếu, notebook tạo token tạm thời và xóa token đó khi dọn dẹp.
5. Notebook dùng Cloudflare Quick Tunnel để kiểm tra transport tùy chọn. Với Named Tunnel cố định, cấu hình CLI bằng EXPOSURE_MODE=named, TUNNEL_TOKEN và EXPOSURE_PUBLIC_URL; xem hướng dẫn agent-server song ngữ.

> **Qualification scope / Phạm vi qualification:** This notebook checks source identity, T4 x2, HF-only target/draft/projector integrity, effective context and output limits, authenticated API, tool-compatible chat payload handling, non-stream and SSE generations, optional Quick Tunnel, evidence and cleanup. A successful Kaggle run qualifies only the observed run; it does not promise 24/7 availability or a fixed Context if GPU memory forces an automatic fallback.

## 1. Configure the canonical production notebook contract / Cấu hình contract canonical của notebook production

**English:** This cell defines the expected version and fork-source identity, Kaggle working paths, safety defaults, helper functions, canonical serving expectations, the agent request policy (`reasoning_strength=high`), and the requested 131072 Context/output target with adaptive fallback. It does not start the model. Expected output begins with `CONFIG=READY` and prints only non-secret configuration values.

**Tiếng Việt:** Cell này khai báo danh tính public release đã khóa, các đường dẫn làm việc trên Kaggle, mặc định an toàn, các hàm hỗ trợ, cấu hình serving canonical, chính sách request agent (`reasoning_strength=high`) cùng mục tiêu Context/output 131072 và fallback thích ứng. Cell này chưa khởi động model. Output mong đợi bắt đầu bằng `CONFIG=READY` và chỉ in các giá trị cấu hình không nhạy cảm.


In [ ]:
from pathlib import Path
import os, json, subprocess, shutil, hashlib, datetime, getpass, urllib.parse, secrets, signal, socket, time, zipfile

REPO = "chloerbeach2026/Muse-Glimmer-30B-GGUF-DFlash2-Kaggle-GPU-T4x2"
REPO_URL = f"https://github.com/{REPO}.git"
SOURCE_REF = "main"

EXPECTED_VERSION = "1.0.0"

WORKDIR = Path("/kaggle/working/muse-glimmer-30b-v1.0.0")
BIN_DIR = Path("/kaggle/working/.muse-demo-bin")
EVIDENCE_PATH = Path("/kaggle/working/muse-glimmer-30b-v1.0.0-demo-evidence.json")
INTERACTION_EVIDENCE_PATH = Path("/kaggle/working/muse-glimmer-30b-v1.0.0-interaction-evidence.json")
REVIEW_SUMMARY_PATH = Path("/kaggle/working/muse-glimmer-30b-v1.0.0-review-summary.txt")
SHA256SUMS_PATH = Path("/kaggle/working/muse-glimmer-30b-v1.0.0-SHA256SUMS")
PUBLICATION_EVIDENCE_ZIP_PATH = Path("/kaggle/working/muse-glimmer-30b-v1.0.0-publication-evidence.zip")
PUBLICATION_EVIDENCE_ZIP_SHA256_PATH = Path(str(PUBLICATION_EVIDENCE_ZIP_PATH) + ".sha256")
GENERATED_SECRET_DIR = Path("/kaggle/working/.muse-secrets")
GENERATED_TOKEN_FILE = GENERATED_SECRET_DIR / "MUSE_API_TOKEN"
FORENSIC_DIR = Path("/kaggle/working/muse-glimmer-30b-v1.0.0-forensic")
FORENSIC_ZIP_PATH = Path("/kaggle/working/muse-glimmer-30b-v1.0.0-forensic.zip")

RUN_NONSTREAM_DEMO = True
RUN_SSE_DEMO = True
RUN_SEMANTIC_SANITY = True
VERIFY_FULL_MODEL_SHA256 = True
AUTO_CLEANUP_AT_END = True
AUTO_DELETE_GENERATED_TOKEN_AT_END = True
PUBLIC_DNS_RETRY_SECONDS = 90
PUBLIC_RETRY_INTERVAL_SECONDS = 2
PUBLIC_HTTP_TIMEOUT_SECONDS = 12
DEMO_REQUEST_MAX_TOKENS = 2048
DEMO_REASONING_STRENGTH = "high"
DUAL_GPU_MIN_RESIDENCY_DELTA_MIB = 512
MODEL_HASH_CHUNK_MIB = 16

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["SERVE_PROFILE"] = "dflash2"
os.environ["EXPOSURE_MODE"] = "quick"
os.environ["PYTHONUNBUFFERED"] = "1"

RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

SESSION = {
    "notebook_contract": "Muse-Glimmer-30B HF-only agent-server qualification",
    "notebook_schema": "muse-glimmer-30b.agent-server-qualification.v4",
    "run_id": RUN_ID,
    "source_ref": SOURCE_REF,
    "expected_version": EXPECTED_VERSION,
    "started_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "forensic_capture_count": 0,
}

def run(args, *, cwd=None, env=None, capture=True, check=True):
    if isinstance(args, str):
        raise TypeError("run() requires a list/tuple, not a shell string")
    p = subprocess.run(
        list(args),
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
        check=False,
    )
    if capture and p.stdout:
        print(p.stdout.rstrip())
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed rc={p.returncode}: {args!r}")
    return p




def classify_muse_demo_process(cmdline):
    """Return a narrow Muse demo process class, or None for anything ambiguous."""
    cmdline = (cmdline or "").strip()
    if not cmdline:
        return None

    cloudflared_candidates = {str(BIN_DIR / "cloudflared")}
    selected_cloudflared = os.environ.get("CLOUDFLARED_BIN")
    if selected_cloudflared:
        cloudflared_candidates.add(str(Path(selected_cloudflared)))

    if (
        any(cmdline.startswith(candidate + " ") for candidate in cloudflared_candidates)
        and " tunnel " in f" {cmdline} "
        and "--url" in cmdline
        and "http://127.0.0.1:8090" in cmdline
    ):
        return "cloudflared"

    auth_script = str(WORKDIR / "scripts" / "auth_proxy.py")
    if (
        auth_script in cmdline
        and "--listen-host 127.0.0.1" in cmdline
        and "--listen-port 8090" in cmdline
        and "--backend-host 127.0.0.1" in cmdline
        and "--backend-port 8088" in cmdline
    ):
        return "auth_proxy"

    if (
        "llama-server" in cmdline
        and "Muse-Glimmer-30B-Q4_K_M.gguf" in cmdline
        and "-a muse-glimmer-30B" in cmdline
        and "--host 127.0.0.1" in cmdline
        and "--port 8088" in cmdline
    ):
        return "llama_server"

    return None


def list_muse_demo_processes():
    p = subprocess.run(
        ["ps", "-eo", "pid=,args="],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    found = []
    if p.returncode != 0:
        return found
    for raw in p.stdout.splitlines():
        raw = raw.strip()
        if not raw:
            continue
        parts = raw.split(None, 1)
        if len(parts) != 2 or not parts[0].isdigit():
            continue
        pid = int(parts[0])
        if pid == os.getpid():
            continue
        kind = classify_muse_demo_process(parts[1])
        if kind:
            found.append((pid, kind))
    return found


def pid_is_alive(pid):
    try:
        stat_path = Path(f"/proc/{pid}/stat")
        if stat_path.is_file():
            fields = stat_path.read_text(encoding="utf-8", errors="replace").split()
            if len(fields) >= 3 and fields[2] == "Z":
                return False
        os.kill(pid, 0)
        return True
    except (ProcessLookupError, FileNotFoundError):
        return False
    except PermissionError:
        return True


def terminate_muse_demo_processes(processes, timeout_seconds=5.0):
    """Terminate only already-classified exact Muse demo PIDs."""
    unique = []
    seen = set()
    for pid, kind in processes:
        if pid in seen:
            continue
        seen.add(pid)
        unique.append((pid, kind))

    for pid, kind in unique:
        try:
            os.kill(pid, signal.SIGTERM)
            print(f"STALE_PROCESS_SIGTERM pid={pid} kind={kind}")
        except ProcessLookupError:
            pass

    deadline = time.monotonic() + float(timeout_seconds)
    while time.monotonic() < deadline:
        if not any(pid_is_alive(pid) for pid, _ in unique):
            break
        time.sleep(0.1)

    killed = []
    for pid, kind in unique:
        if not pid_is_alive(pid):
            continue
        try:
            os.kill(pid, signal.SIGKILL)
            killed.append((pid, kind))
            print(f"STALE_PROCESS_SIGKILL pid={pid} kind={kind}")
        except ProcessLookupError:
            pass

    return {
        "matched": len(unique),
        "sigkill_count": len(killed),
    }


def loopback_port_is_free(port):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(0.25)
    try:
        return s.connect_ex(("127.0.0.1", int(port))) != 0
    finally:
        s.close()



def query_gpu_snapshot(label):
    """Return reviewer-safe NVIDIA GPU telemetry without depending on NVML Python packages."""
    result = run([
        "nvidia-smi",
        "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
        "--format=csv,noheader,nounits",
    ])
    rows = []
    for raw in result.stdout.strip().splitlines():
        parts = [x.strip() for x in raw.split(",")]
        if len(parts) != 5:
            raise RuntimeError(f"Unexpected nvidia-smi row: {raw!r}")
        index, name, memory_used, memory_total, utilization = parts
        row = {
            "index": int(index),
            "name": name,
            "memory_used_mib": int(memory_used),
            "memory_total_mib": int(memory_total),
            "utilization_gpu_percent": int(utilization),
        }
        rows.append(row)
    rows.sort(key=lambda x: x["index"])
    print(f"GPU_SNAPSHOT_BEGIN label={label}")
    for row in rows:
        print(
            f"GPU_{row['index']} name={row['name']} "
            f"memory_used_mib={row['memory_used_mib']} "
            f"memory_total_mib={row['memory_total_mib']} "
            f"utilization_gpu_percent={row['utilization_gpu_percent']}"
        )
    print(f"GPU_SNAPSHOT_END label={label}")
    return rows


def sha256_file_streaming(path, chunk_mib=16):
    """Hash a large Hugging Face model file without loading it into RAM."""
    path = Path(path)
    h = hashlib.sha256()
    chunk_size = int(chunk_mib) * 1024 * 1024
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def verify_auth_proxy_runtime_limits(context_size, output_tokens):
    """Verify the running gateway uses the actual Context/output limits."""
    matches = [pid for pid, kind in list_muse_demo_processes() if kind == "auth_proxy"]
    assert len(matches) == 1, f"Expected exactly one Muse auth_proxy process, found {matches}"
    pid = matches[0]
    raw = Path(f"/proc/{pid}/cmdline").read_bytes()
    argv = [part.decode("utf-8", errors="replace") for part in raw.split(b"\\0") if part]
    auth_proxy_path = str(WORKDIR / "scripts" / "auth_proxy.py")
    assert auth_proxy_path in argv, "Running auth proxy does not use this fork checkout."
    def arg_value(flag):
        assert flag in argv, f"Auth proxy is missing {flag}."
        idx = argv.index(flag)
        assert idx + 1 < len(argv), f"Missing value after {flag}."
        return int(argv[idx + 1])
    actual_context = arg_value("--effective-context-size")
    actual_output = arg_value("--max-output-tokens")
    assert actual_context == int(context_size), f"Gateway Context mismatch {actual_context} != {context_size}"
    assert actual_output == int(output_tokens), f"Gateway output cap mismatch {actual_output} != {output_tokens}"
    assert arg_value("--queue-size") == 8, "Gateway bounded queue must be 8 requests."
    print("GATEWAY_RUNTIME_LIMITS_PROCESS_GATE=PASS")
    print(f"AUTH_PROXY_PID={pid}")
    print(f"GATEWAY_EFFECTIVE_CONTEXT_SIZE={actual_context}")
    print(f"GATEWAY_EFFECTIVE_MAX_OUTPUT_TOKENS={actual_output}")
    return pid


def repo_config_keys():
    # Parse the pinned common.sh _CONFIG_KEYS block instead of hardcoding it.
    common = WORKDIR / "scripts" / "common.sh"
    text = common.read_text(encoding="utf-8", errors="replace")
    start = text.find("_CONFIG_KEYS=(")
    if start < 0:
        raise RuntimeError("Could not locate _CONFIG_KEYS in scripts/common.sh")
    start += len("_CONFIG_KEYS=(")
    end = text.find("\n)", start)
    if end < 0:
        raise RuntimeError("Could not parse _CONFIG_KEYS block in scripts/common.sh")
    import shlex
    return tuple(shlex.split(text[start:end]))


REPO_DERIVED_ENV_KEYS = {
    "VENV_DIR", "LLAMA_DIR", "MODELS_DIR", "RUNS_DIR", "ARTIFACTS_DIR",
    "COMPARISONS_DIR", "RUNTIME_STATE_DIR",
    "LLAMA_RESOLVED_BIN_DIR", "LLAMA_RUNTIME_SOURCE",
    "LLAMA_RUNTIME_LD_LIBRARY_PATH", "LLAMA_RUNTIME_DATASET_ROOT",
    "LLAMA_RUNTIME_VERSION_TEXT", "LLAMA_SOURCE_SNAPSHOT",
    "LLAMA_SOURCE_PATH", "LLAMA_SOURCE_TREE_SHA256", "LLAMA_BUILD_CMAKE_ARGS",
    "LLAMA_BUILD_COMPILER", "LLAMA_BUILD_CUDA_VERSION", "LLAMA_BUILD_DIR",
    "LLAMA_BINARY_SHA256_SERVER", "LLAMA_BINARY_SHA256_CLI",
    "LLAMA_BINARY_SHA256_BENCH", "LLAMA_RUNTIME_MANIFEST_VERIFIED",
    "LLAMA_RUNTIME_MANIFEST_PATH", "LLAMA_RUNTIME_SOURCE_COMMIT",
    "MODEL_RESOLVED_PATH", "MODEL_RESOLVED_FILE", "MODEL_RUNTIME_SOURCE",
    "MODEL_RUNTIME_DATASET_ROOT", "MODEL_RESOLVED_SIZE_BYTES",
    "MODEL_RESOLVED_SHA256", "MODEL_METADATA_SOURCE", "MODEL_SELECTION",
    "MODEL_VERIFICATION_SECONDS", "SETUP_TOTAL_SECONDS",
    "VENV_BOOTSTRAP_SECONDS", "RUNTIME_DISCOVERY_SECONDS", "VENV_REUSED",
    "LLAMA_SOURCE_BUILD_SECONDS", "SOURCE_BUILD_CACHE_HIT",
    "CUDAToolkit_ROOT", "CUDACXX", "CMAKE_CUDA_COMPILER",
    "CUDA_TOOLKIT_ROOT_RESOLVED", "CUDA_NVCC_PATH", "CUDA_NVCC_REALPATH",
    "CUDA_RUNTIME_HEADER", "CUDA_CUDART_PATH", "CUDA_TOOLKIT_VERSION",
    "CUDA_DRIVER_LIBRARY", "CUDA_PATH",
    "NVIDIA_SMI_PATH", "NVIDIA_SMI_REALPATH", "NVIDIA_SMI_BIN_DIR",
    "NVIDIA_ML_LIBRARY", "NVIDIA_DRIVER_LIB_DIR",
    "NVIDIA_RUNTIME_LD_LIBRARY_PATH", "NVIDIA_RUNTIME_PATH_INJECTED",
    "NVIDIA_RUNTIME_LD_LIBRARY_INJECTED",
    "EXTERNAL_MODE", "TUNNEL_TOKEN", "EXPOSURE_PUBLIC_URL",
    "MUSE_DEMO_TELEMETRY_FILE", "EXTERNAL_LOG_FILE",
}


def build_canonical_demo_env(token_value, cloudflared_path, exposure_mode="proxy"):
    # Preserve Kaggle/system env, remove only repository-owned stale overrides.
    child = os.environ.copy()
    removed = []

    for key in set(repo_config_keys()) | REPO_DERIVED_ENV_KEYS:
        if key in child:
            removed.append(key)
            child.pop(key, None)

    for key in list(child):
        if key.startswith("EXPOSURE_") or key.startswith("EXTERNAL_"):
            removed.append(key)
            child.pop(key, None)

    child["CUDA_VISIBLE_DEVICES"] = "0,1"
    child["SERVE_PROFILE"] = "dflash2"
    if exposure_mode not in {"proxy", "quick", "named"}:
        raise ValueError(f"Unsupported exposure_mode={exposure_mode!r}")
    child["EXPOSURE_MODE"] = exposure_mode
    child["PYTHONUNBUFFERED"] = "1"
    child["MUSE_API_TOKEN"] = token_value
    child["CLOUDFLARED_BIN"] = str(cloudflared_path)
    return child, sorted(set(removed))



def gateway_policy_max_tokens(child_env):
    # Read the cap from the pinned v1.0.0 source itself.
    script = (
        "import sys;"
        "sys.path.insert(0,'scripts');"
        "import demo_policy as P;"
        "print(P.MAX_TOKENS)"
    )
    result = run(
        ["python3", "-B", "-c", script],
        cwd=WORKDIR,
        env=child_env,
        check=True,
    )
    value = int(result.stdout.strip().splitlines()[-1])
    return value

def canonical_effective_config(child_env):
    # Resolve exactly like persistent serve.sh: RUN_PROFILE=agent.
    keys = [
        "RUN_PROFILE", "CUDA_VISIBLE_DEVICES",
        "GPU_LAYERS", "GPU_SPLIT_MODE", "GPU_TENSOR_SPLIT",
        "PARALLEL_SLOTS", "CONTEXT_SIZE", "BATCH_SIZE", "UBATCH_SIZE",
        "DFLASH_DRAFT_N_MAX", "DFLASH_DRAFT_GPU_LAYERS",
        "MAX_TOKENS", "REASONING_STRENGTH", "REASONING_BUDGET",
        "MIN_ANSWER_TOKENS", "PROMPT_LIMIT",
    ]
    shell = r'''
set -Eeuo pipefail
source scripts/common.sh
RUN_PROFILE=agent
load_profile_exports
for k in RUN_PROFILE CUDA_VISIBLE_DEVICES GPU_LAYERS GPU_SPLIT_MODE GPU_TENSOR_SPLIT \
         PARALLEL_SLOTS CONTEXT_SIZE BATCH_SIZE UBATCH_SIZE DFLASH_DRAFT_N_MAX \
         DFLASH_DRAFT_GPU_LAYERS MAX_TOKENS REASONING_STRENGTH REASONING_BUDGET \
         MIN_ANSWER_TOKENS PROMPT_LIMIT
do
  printf '%s=%s\n' "$k" "${!k-}"
done
'''
    result = run(
        ["bash", "-lc", shell],
        cwd=WORKDIR,
        env=child_env,
        check=True,
    )
    resolved = parse_env_blob(result.stdout)
    return {k: resolved.get(k) for k in keys}


EXPECTED_CANONICAL_EFFECTIVE_CONFIG = {
    "RUN_PROFILE": "agent",
    "CUDA_VISIBLE_DEVICES": "0,1",
    "GPU_LAYERS": "999",
    "GPU_SPLIT_MODE": "layer",
    "GPU_TENSOR_SPLIT": "1,1",
    "PARALLEL_SLOTS": "1",
    "CONTEXT_SIZE": "131072",
    "BATCH_SIZE": "512",
    "UBATCH_SIZE": "128",
    "DFLASH_DRAFT_N_MAX": "15",
    "DFLASH_DRAFT_GPU_LAYERS": "999",
    "MAX_TOKENS": "131072",
    "REASONING_STRENGTH": "high",
    "REASONING_BUDGET": "0",
    "MIN_ANSWER_TOKENS": "1024",
    "PROMPT_LIMIT": "0",
}


def read_tail(path, max_lines=200):
    path = Path(path)
    if not path.is_file():
        return ""
    return "\n".join(
        path.read_text(encoding="utf-8", errors="replace").splitlines()[-max_lines:]
    ) + "\n"


def safe_copy_text(src, dst, max_lines=None):
    src = Path(src)
    dst = Path(dst)
    if not src.is_file():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    if max_lines is None:
        text = src.read_text(encoding="utf-8", errors="replace")
    else:
        text = read_tail(src, max_lines=max_lines)
    dst.write_text(text, encoding="utf-8")
    return True


def capture_backend_forensics(stage, response=None, env=None):
    # Capture bounded operational evidence only; never write bearer tokens.
    SESSION["forensic_capture_count"] = int(SESSION.get("forensic_capture_count", 0)) + 1
    child_env = (env or os.environ.copy()).copy()
    child_env.pop("TUNNEL_TOKEN", None)

    stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    stage_dir = FORENSIC_DIR / f"{stamp}-{stage}"
    stage_dir.mkdir(parents=True, exist_ok=True)

    summary = {
        "stage": stage,
        "captured_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "response_http": getattr(response, "status_code", None),
        "response_error_code": None,
        "response_error_type": None,
        "response_request_id": None,
        "port_8088_open": not loopback_port_is_free(8088),
        "port_8090_open": not loopback_port_is_free(8090),
        "muse_processes": [
            {"pid": pid, "kind": kind}
            for pid, kind in list_muse_demo_processes()
        ],
    }

    if response is not None:
        try:
            payload = response.json()
            err = payload.get("error") or {}
            summary["response_error_code"] = err.get("code")
            summary["response_error_type"] = err.get("type")
            summary["response_request_id"] = (
                payload.get("request_id") or err.get("request_id")
            )
        except Exception:
            pass

    commands = {
        "external-status.env": ["./external.sh", "status", "--format", "env"],
        "serve-status.txt": ["./serve.sh", "status"],
        "nvidia-smi.txt": [
            "nvidia-smi",
            "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
            "--format=csv,noheader,nounits",
        ],
    }

    for filename, args in commands.items():
        result = run(
            args,
            cwd=WORKDIR if args[0].startswith("./") else None,
            env=child_env,
            check=False,
        )
        (stage_dir / filename).write_text(
            result.stdout or "",
            encoding="utf-8",
        )

    (stage_dir / "muse-processes.txt").write_text(
        "".join(
            f"pid={item['pid']} kind={item['kind']}\n"
            for item in summary["muse_processes"]
        ),
        encoding="utf-8",
    )

    state_dir = WORKDIR / "artifacts" / "runtime-state"

    for name in (
        "server.json",
        "server-metadata.json",
        "exposure.json",
        "demo-observability.env",
        "demo-observability.json",
        "llama.env",
        "model.env",
        "setup.env",
        "nvidia.env",
    ):
        safe_copy_text(state_dir / name, stage_dir / name)

    for name in (
        "server.log",
        "exposure-proxy.log",
        "exposure-tunnel.log",
        "external.log",
    ):
        safe_copy_text(
            state_dir / name,
            stage_dir / f"{name}.tail.txt",
            max_lines=250,
        )

    (stage_dir / "forensic-summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    (FORENSIC_DIR / "latest-summary.json").write_text(
        json.dumps(summary, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )

    if FORENSIC_ZIP_PATH.exists():
        FORENSIC_ZIP_PATH.unlink()
    shutil.make_archive(
        str(FORENSIC_ZIP_PATH.with_suffix("")),
        "zip",
        root_dir=FORENSIC_DIR,
    )

    print(f"FORENSIC_CAPTURE=PASS stage={stage}")
    print(f"FORENSIC_DIR={stage_dir}")
    print(f"FORENSIC_ZIP={FORENSIC_ZIP_PATH}")
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    return summary

def parse_env_blob(text):
    out = {}
    for raw in text.splitlines():
        if "=" not in raw:
            continue
        key, value = raw.split("=", 1)
        key = key.strip()
        if key:
            out[key] = value.strip()
    return out

print("CONFIG=READY")
print(f"SOURCE_REF={SOURCE_REF}")
print(f"EXPECTED_VERSION={EXPECTED_VERSION}")
print(f"RUN_ID={RUN_ID}")
print(f"AUTO_CLEANUP_AT_END={AUTO_CLEANUP_AT_END}")
print(f"AUTO_DELETE_GENERATED_TOKEN_AT_END={AUTO_DELETE_GENERATED_TOKEN_AT_END}")
print(f"PUBLIC_DNS_RETRY_SECONDS={PUBLIC_DNS_RETRY_SECONDS}")
print(f"DEMO_REQUEST_MAX_TOKENS={DEMO_REQUEST_MAX_TOKENS}")
print(f"DEMO_REASONING_STRENGTH={DEMO_REASONING_STRENGTH}")
print(f"RUN_SEMANTIC_SANITY={RUN_SEMANTIC_SANITY}")
print(f"VERIFY_FULL_MODEL_SHA256={VERIFY_FULL_MODEL_SHA256}")
print(f"DUAL_GPU_MIN_RESIDENCY_DELTA_MIB={DUAL_GPU_MIN_RESIDENCY_DELTA_MIB}")


## Operator lifecycle controls / Điều khiển vòng đời cho operator

**English.** Normal use is simply **Run All**. The notebook already performs automatic exact-process recovery before every start. The helpers below are for interrupted runs, manual testing, or when `AUTO_CLEANUP_AT_END=False`.

* `muse_stop()` stops only the canonical tunnel/proxy/backend processes and keeps the checkout/evidence.
* `muse_reset(preserve_evidence=True)` additionally removes notebook-owned runtime/source state so the same notebook can be run again from the top.
* `muse_reset(preserve_evidence=False)` also removes the previous publication evidence bundle.
* Unknown listeners on ports `8088`/`8090` are never killed; the helper fails closed.

**Tiếng Việt.** Cách dùng bình thường chỉ cần **Run All**. Notebook vốn đã tự recovery các process khớp chính xác trước mỗi lần start. Các helper dưới đây dành cho run bị gián đoạn, test thủ công hoặc khi `AUTO_CLEANUP_AT_END=False`.

* `muse_stop()` chỉ dừng tunnel/proxy/backend canonical và giữ checkout/evidence.
* `muse_reset(preserve_evidence=True)` xóa thêm runtime/source state do notebook sở hữu để có thể chạy lại chính notebook này từ đầu.
* `muse_reset(preserve_evidence=False)` xóa thêm publication evidence bundle của run trước.
* Không bao giờ kill listener không xác định trên port `8088`/`8090`; helper sẽ fail closed.


In [ ]:
def muse_stop(delete_generated_token=True):
    """Stop only exact Muse demo processes; fail closed on foreign listeners."""
    canonical_stop_attempted = False
    canonical_stop_rc = None
    external = WORKDIR / "external.sh"

    if external.is_file() and os.access(external, os.X_OK):
        canonical_stop_attempted = True
        stop_env = os.environ.copy()
        if (
            "build_canonical_demo_env" in globals()
            and globals().get("token")
            and globals().get("cloudflared")
        ):
            try:
                stop_env = build_canonical_demo_env(
                    token, cloudflared, exposure_mode="quick"
                )[0]
            except Exception as exc:
                print(f"LIFECYCLE_CANONICAL_ENV_FALLBACK={type(exc).__name__}")
        stop_result = run(
            ["./external.sh", "stop-all"],
            cwd=WORKDIR,
            env=stop_env,
            check=False,
        )
        canonical_stop_rc = stop_result.returncode
        print(f"LIFECYCLE_CANONICAL_STOP_RC={canonical_stop_rc}")

    time.sleep(0.5)
    matched = list_muse_demo_processes()
    termination = {"matched": 0, "sigkill_count": 0}
    if matched:
        print(f"LIFECYCLE_EXACT_PROCESS_MATCHES={len(matched)}")
        termination = terminate_muse_demo_processes(matched, timeout_seconds=5.0)

    time.sleep(0.25)
    remaining = list_muse_demo_processes()
    assert not remaining, f"Exact Muse processes survived stop: {remaining}"

    free_8088 = loopback_port_is_free(8088)
    free_8090 = loopback_port_is_free(8090)
    print(f"LIFECYCLE_PORT_8088_FREE={'PASS' if free_8088 else 'FAIL'}")
    print(f"LIFECYCLE_PORT_8090_FREE={'PASS' if free_8090 else 'FAIL'}")
    assert free_8088, "Port 8088 is occupied by an unknown/foreign listener; refusing to kill it."
    assert free_8090, "Port 8090 is occupied by an unknown/foreign listener; refusing to kill it."

    token_deleted = False
    if delete_generated_token and GENERATED_TOKEN_FILE.exists():
        GENERATED_TOKEN_FILE.unlink()
        token_deleted = True
        try:
            if GENERATED_SECRET_DIR.exists() and not any(GENERATED_SECRET_DIR.iterdir()):
                GENERATED_SECRET_DIR.rmdir()
        except OSError:
            pass

    summary = {
        "canonical_stop_attempted": canonical_stop_attempted,
        "canonical_stop_rc": canonical_stop_rc,
        "matched_exact_processes": termination["matched"],
        "sigkill_count": termination["sigkill_count"],
        "port_8088_free": free_8088,
        "port_8090_free": free_8090,
        "generated_token_deleted": token_deleted,
    }
    print("MUSE_STOP=PASS")
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    return summary


def muse_reset(preserve_evidence=True):
    """Stop the demo and remove only notebook-owned state so this notebook can be re-run."""
    summary = muse_stop(delete_generated_token=True)

    removed = []
    for path in (WORKDIR, FORENSIC_DIR):
        if path.exists():
            shutil.rmtree(path)
            removed.append(str(path))
    if FORENSIC_ZIP_PATH.exists():
        FORENSIC_ZIP_PATH.unlink()
        removed.append(str(FORENSIC_ZIP_PATH))

    if not preserve_evidence:
        for path in (
            EVIDENCE_PATH,
            INTERACTION_EVIDENCE_PATH,
            REVIEW_SUMMARY_PATH,
            SHA256SUMS_PATH,
            PUBLICATION_EVIDENCE_ZIP_PATH,
            PUBLICATION_EVIDENCE_ZIP_SHA256_PATH,
        ):
            if path.exists():
                path.unlink()
                removed.append(str(path))

    summary["removed_paths"] = removed
    summary["preserve_evidence"] = bool(preserve_evidence)
    print("MUSE_RESET=PASS")
    print("RERUN_READY=PASS")
    print("Run the notebook again from the first cell; a new notebook is not required.")
    return summary


print("LIFECYCLE_CONTROLS=READY")
print("NORMAL_RERUN=Run All again")
print("INTERRUPTED_RUN_STOP=muse_stop()")
print("INTERRUPTED_RUN_RESET=muse_reset(preserve_evidence=True)")


## 2. Verify the Kaggle NVIDIA T4 x2 hardware gate / Xác minh cổng phần cứng Kaggle NVIDIA T4 x2

**English:** This cell verifies that the notebook is running on Kaggle with exactly two NVIDIA T4 GPUs and at least 14,000 MiB visible VRAM per GPU. A successful qualification ends with `KAGGLE_T4X2_GATE=PASS`. If this gate fails, do not continue with the real production-demo run.

**Tiếng Việt:** Cell này xác minh notebook đang chạy trên Kaggle với đúng hai GPU NVIDIA T4 và mỗi GPU có ít nhất 14.000 MiB VRAM khả dụng. Khi đạt yêu cầu, output kết thúc bằng `KAGGLE_T4X2_GATE=PASS`. Nếu gate này FAIL thì không nên tiếp tục real production-demo.


In [ ]:
assert Path("/kaggle/working").is_dir(), "This notebook is intended for Kaggle."
gpus = query_gpu_snapshot("hardware-gate")

assert len(gpus) == 2, f"Expected exactly 2 GPUs, found {len(gpus)}"
assert all("T4" in x["name"] for x in gpus), f"Expected NVIDIA T4 x2, found {gpus}"
assert all(x["memory_total_mib"] >= 14000 for x in gpus), f"Each GPU must expose >=14000 MiB, found {gpus}"

SESSION["gpus"] = gpus
SESSION["gpu_topology_gate"] = "PASS"
print("KAGGLE_T4X2_GATE=PASS")


## 3. Configure the Hugging Face-only model assets / Cấu hình model assets chỉ tải từ Hugging Face

**English:** The persistent server downloads the target, DFlash2 draft and vision projector from pinned Hugging Face revisions. This cell declares exact local filenames, immutable revisions, expected file sizes and SHA-256 digests. It does not read model weights from /kaggle/input; the post-start gate checks the verification records written by the downloader before accepting the model/runtime as ready.

**Tiếng Việt:** Server tải target, draft DFlash2 và vision projector từ revision Hugging Face đã pin. Cell này khai báo tên file local, revision bất biến, dung lượng và SHA-256 mong đợi. Cell không đọc model từ /kaggle/input; sau khi khởi động, gate kiểm tra các bản ghi xác minh do downloader ghi trước khi chấp nhận model/runtime.

In [ ]:
TARGET_NAME = "Muse-Glimmer-30B-Q4_K_M.gguf"
DRAFT_NAME = "Muse-Glimmer-30B-DFlash2-Q4_K_M.gguf"
MM_PROJ_NAME = "mmproj-Muse-Glimmer-30B-Q4_K_M.gguf"

TARGET_HF_REPO = "bartowski/Muse-Glimmer-30B-GGUF"
TARGET_HF_REVISION = "16f5840bef28e4018cf41a7ec9709d6fd89176f6"
TARGET_EXPECTED_SIZE_BYTES = 17306324000
TARGET_EXPECTED_SHA256 = "0d3fc85f61d10fdc84072f0bba6005d61c1ac5605a2627b0fd5ea4ff8194c384"

DRAFT_HF_REPO = "meta-models/Muse-Glimmer-30B-GGUF"
DRAFT_HF_REVISION = "43c7eadd41352a299ea8e0a36b3157978dd63596"
DRAFT_HF_FILE = "dflash-Muse-Glimmer-30B-Q4_K_M.gguf"
DRAFT_EXPECTED_SIZE_BYTES = 1631208128
DRAFT_EXPECTED_SHA256 = "b2e808bf656086fe86bd0d0bd990f01d33e377537a07c02d45371517c8b264ef"

MM_PROJ_HF_REPO = "meta-models/Muse-Glimmer-30B-GGUF"
MM_PROJ_HF_REVISION = "43c7eadd41352a299ea8e0a36b3157978dd63596"
MM_PROJ_EXPECTED_SIZE_BYTES = 1400328928
MM_PROJ_EXPECTED_SHA256 = "f48b452316f9b213758e8659444029b961a24a07f99a1abb2a9f88b06f7c00c6"

MODEL_DIR = WORKDIR / "models"
TARGET_PATH = MODEL_DIR / TARGET_NAME
DRAFT_PATH = MODEL_DIR / DRAFT_NAME
MM_PROJ_PATH = MODEL_DIR / MM_PROJ_NAME

SESSION["model_source_mode"] = "hf"
SESSION["target_hf_repo"] = TARGET_HF_REPO
SESSION["target_hf_revision"] = TARGET_HF_REVISION
SESSION["draft_hf_repo"] = DRAFT_HF_REPO
SESSION["draft_hf_revision"] = DRAFT_HF_REVISION
SESSION["mmproj_hf_repo"] = MM_PROJ_HF_REPO
SESSION["mmproj_hf_revision"] = MM_PROJ_HF_REVISION
SESSION["target_expected_sha256"] = TARGET_EXPECTED_SHA256
SESSION["draft_expected_sha256"] = DRAFT_EXPECTED_SHA256
SESSION["mmproj_expected_sha256"] = MM_PROJ_EXPECTED_SHA256
SESSION["target_expected_size_bytes"] = TARGET_EXPECTED_SIZE_BYTES
SESSION["draft_expected_size_bytes"] = DRAFT_EXPECTED_SIZE_BYTES
SESSION["mmproj_expected_size_bytes"] = MM_PROJ_EXPECTED_SIZE_BYTES
SESSION["target_observed_sha256"] = None
SESSION["draft_observed_sha256"] = None
SESSION["mmproj_observed_sha256"] = None
SESSION["target_sha256_seconds"] = None
SESSION["draft_sha256_seconds"] = None
SESSION["model_sha256_gate"] = "PENDING_HF_DOWNLOAD"

print("HF_MODEL_SOURCE=PINNED")
print(f"HF_TARGET_REVISION={TARGET_HF_REVISION}")
print(f"HF_DFLASH_REVISION={DRAFT_HF_REVISION}")
print(f"HF_MMPROJ_REVISION={MM_PROJ_HF_REVISION}")
print("HF_MODEL_INTEGRITY_GATE=PENDING_SERVER_START")


## 4. Recover safely, clear stale failure residue, and verify this fork's source / Khôi phục an toàn, dọn residue lỗi cũ và xác minh nguồn của fork này

**English:** Before deleting any previous worktree, this cell attempts canonical `stop-all`, scans only exact Muse demo process signatures, terminates only matched owned processes, and refuses to kill unknown listeners on ports 8088/8090. After the process/port gates are clean, it removes forensic artifacts left by an earlier run so evidence from the new qualification cannot be confused with stale failure data. It then resolves and clones this fork's `main` branch, verifies checkout HEAD against the exact remote commit SHA resolved immediately before cloning, checks `VERSION=1.0.0` and a clean worktree, then validates `SOURCE_MANIFEST.sha256`. Key PASS lines include `RERUN_RECOVERY=PASS`, `STALE_FORENSIC_CLEANUP=PASS`, and `FORK_SOURCE_IDENTITY=PASS`.

**Tiếng Việt:** Trước khi xóa worktree cũ, cell này thử canonical `stop-all`, chỉ quét các process signature chính xác của Muse demo, chỉ dừng process đã nhận diện là thuộc demo và từ chối kill listener lạ trên port 8088/8090. Sau khi process/port gate sạch, cell xóa forensic artifact còn sót từ run trước để evidence của lần qualification mới không bị nhầm với dữ liệu lỗi cũ. Sau đó cell resolve và clone nhánh `main` của fork này, xác minh HEAD khớp SHA commit remote đã resolve ngay trước khi clone, kiểm tra `VERSION=1.0.0`, worktree sạch và `SOURCE_MANIFEST.sha256`. Các dòng PASS quan trọng gồm `RERUN_RECOVERY=PASS`, `STALE_FORENSIC_CLEANUP=PASS` và `FORK_SOURCE_IDENTITY=PASS`.


In [ ]:
# Safe rerun recovery MUST happen before deleting the old worktree/state.
canonical_stop_attempted = False
canonical_stop_rc = None

previous_external = WORKDIR / "external.sh"
if previous_external.is_file() and os.access(previous_external, os.X_OK):
    canonical_stop_attempted = True
    print("RERUN_RECOVERY_CANONICAL_STOP=ATTEMPT")
    stop_result = run(
        ["./external.sh", "stop-all", "--format", "env"],
        cwd=WORKDIR,
        env=os.environ.copy(),
        check=False,
    )
    canonical_stop_rc = stop_result.returncode
    print(f"RERUN_RECOVERY_CANONICAL_STOP_RC={canonical_stop_rc}")

# Give a canonical stop a short moment to reap owned children before orphan scan.
time.sleep(0.5)

stale_before = list_muse_demo_processes()
print(f"RERUN_RECOVERY_MATCHED_ORPHANS={len(stale_before)}")
for pid, kind in stale_before:
    print(f"RERUN_RECOVERY_ORPHAN pid={pid} kind={kind}")

termination = {
    "matched": 0,
    "sigkill_count": 0,
}
if stale_before:
    termination = terminate_muse_demo_processes(stale_before, timeout_seconds=5.0)

time.sleep(0.25)
stale_after = list_muse_demo_processes()
assert not stale_after, (
    "Exact Muse demo processes are still alive after bounded recovery: "
    f"{stale_after}"
)

port_8088_free = loopback_port_is_free(8088)
port_8090_free = loopback_port_is_free(8090)

print(f"PORT_8088_FREE_BEFORE_START={'PASS' if port_8088_free else 'FAIL'}")
print(f"PORT_8090_FREE_BEFORE_START={'PASS' if port_8090_free else 'FAIL'}")

assert port_8088_free, (
    "127.0.0.1:8088 is still occupied after exact Muse recovery. "
    "Refusing to kill an unknown/foreign listener."
)
assert port_8090_free, (
    "127.0.0.1:8090 is still occupied after exact Muse recovery. "
    "Refusing to kill an unknown/foreign listener."
)

# Publication hygiene: after exact owned-process recovery and port gates are clean,
# remove forensic artifacts from an earlier run. Any new failure in this run will
# recreate them and increment SESSION["forensic_capture_count"].
stale_forensic_dir_removed = False
stale_forensic_zip_removed = False

if FORENSIC_DIR.exists():
    shutil.rmtree(FORENSIC_DIR)
    stale_forensic_dir_removed = True

if FORENSIC_ZIP_PATH.exists():
    FORENSIC_ZIP_PATH.unlink()
    stale_forensic_zip_removed = True

SESSION["stale_forensic_dir_removed_before_run"] = stale_forensic_dir_removed
SESSION["stale_forensic_zip_removed_before_run"] = stale_forensic_zip_removed
SESSION["stale_forensic_cleanup"] = "PASS"

print(f"STALE_FORENSIC_DIR_REMOVED={stale_forensic_dir_removed}")
print(f"STALE_FORENSIC_ZIP_REMOVED={stale_forensic_zip_removed}")
print("STALE_FORENSIC_CLEANUP=PASS")

# Current-run evidence must never inherit a previous PASS. Remove only publication
# artifacts owned by this notebook; a fresh run will recreate them.
stale_publication_artifacts_removed = []
for artifact_path in (
    EVIDENCE_PATH,
    INTERACTION_EVIDENCE_PATH,
    REVIEW_SUMMARY_PATH,
    SHA256SUMS_PATH,
    PUBLICATION_EVIDENCE_ZIP_PATH,
    PUBLICATION_EVIDENCE_ZIP_SHA256_PATH,
):
    if artifact_path.exists():
        artifact_path.unlink()
        stale_publication_artifacts_removed.append(str(artifact_path))

SESSION["stale_publication_artifacts_removed"] = stale_publication_artifacts_removed
SESSION["stale_publication_cleanup"] = "PASS"
print(f"STALE_PUBLICATION_ARTIFACTS_REMOVED={len(stale_publication_artifacts_removed)}")
print("STALE_PUBLICATION_CLEANUP=PASS")

SESSION["rerun_recovery_canonical_stop_attempted"] = canonical_stop_attempted
SESSION["rerun_recovery_canonical_stop_rc"] = canonical_stop_rc
SESSION["rerun_recovery_orphan_match_count"] = len(stale_before)
SESSION["rerun_recovery_sigkill_count"] = termination["sigkill_count"]
SESSION["port_8088_free_before_start"] = port_8088_free
SESSION["port_8090_free_before_start"] = port_8090_free
SESSION["rerun_recovery"] = "PASS"

print("RERUN_RECOVERY=PASS")

# Only now is it safe to discard the previous source/state directory.
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

remote_ref = run([
    "git", "ls-remote", "--heads", REPO_URL, f"refs/heads/{SOURCE_REF}"
]).stdout.strip().split()
assert len(remote_ref) == 2 and remote_ref[1] == f"refs/heads/{SOURCE_REF}", (
    f"Could not resolve expected fork branch refs/heads/{SOURCE_REF}."
)
resolved_source_sha = remote_ref[0]
assert len(resolved_source_sha) == 40 and all(c in "0123456789abcdef" for c in resolved_source_sha.lower())

run([
    "git", "clone",
    "--depth", "1",
    "--branch", SOURCE_REF,
    REPO_URL,
    str(WORKDIR),
])

head = run(["git", "rev-parse", "HEAD"], cwd=WORKDIR).stdout.strip()
status = run(["git", "status", "--porcelain"], cwd=WORKDIR).stdout.strip()
release_version = (WORKDIR / "VERSION").read_text(encoding="utf-8").strip()

print(f"SOURCE_REF={SOURCE_REF}")
print(f"RESOLVED_SOURCE_SHA={resolved_source_sha}")
print(f"HEAD={head}")
print(f"RELEASE_VERSION={release_version}")
print(f"WORKTREE_CLEAN={status == ''}")

assert head == resolved_source_sha, (
    f"Fork branch moved during clone or checkout drifted: {resolved_source_sha} != {head}"
)
assert release_version == EXPECTED_VERSION, f"VERSION drift: {release_version} != {EXPECTED_VERSION}"
assert status == "", "Fresh clone is unexpectedly dirty."

manifest = run(
    ["python3", "-B", "scripts/source_manifest.py", "check", "--root", "."],
    cwd=WORKDIR,
).stdout
assert "SOURCE_MANIFEST: OK" in manifest

SESSION["source_ref"] = SOURCE_REF
SESSION["resolved_source_sha"] = resolved_source_sha
SESSION["actual_head"] = head
SESSION["release_version"] = release_version
SESSION["worktree_clean"] = True
SESSION["source_manifest"] = "PASS"
print("FORK_SOURCE_IDENTITY=PASS")


## 5. Load or securely generate `MUSE_API_TOKEN` / Nạp hoặc tạo an toàn `MUSE_API_TOKEN`

**English:** This cell first reads the Kaggle Secret `MUSE_API_TOKEN`. If it is missing or invalid, it generates a cryptographically secure temporary token, stores it under `/kaggle/working/.muse-secrets/MUSE_API_TOKEN` with mode `0600`, and never prints the token value. Success is `MUSE_API_TOKEN_VALIDATION=PASS`. Do not publish the generated secret file in notebook outputs.

**Tiếng Việt:** Cell này ưu tiên đọc Kaggle Secret `MUSE_API_TOKEN`. Nếu secret thiếu hoặc không hợp lệ, notebook tự tạo token tạm thời bằng cơ chế mật mã an toàn, lưu tại `/kaggle/working/.muse-secrets/MUSE_API_TOKEN` với mode `0600` và không bao giờ in giá trị token. Thành công là `MUSE_API_TOKEN_VALIDATION=PASS`. Không được publish file secret được tạo tự động trong Notebook Outputs.


In [ ]:
token = None
secret_source = None
GENERATED_TOKEN_CREATED_THIS_RUN = False

def valid_muse_secret(value):
    return (
        isinstance(value, str)
        and len(value) >= 32
        and all(ord(c) >= 32 and ord(c) != 127 for c in value)
    )

# Prefer the operator-managed Kaggle Secret.
try:
    from kaggle_secrets import UserSecretsClient
    candidate = UserSecretsClient().get_secret("MUSE_API_TOKEN")
    if valid_muse_secret(candidate):
        token = candidate
        secret_source = "kaggle_secret"
except Exception:
    token = None

# Safe fallback: generate a temporary secret without ever printing its value.
if token is None:
    GENERATED_SECRET_DIR.mkdir(parents=True, exist_ok=True)
    os.chmod(GENERATED_SECRET_DIR, 0o700)

    token = secrets.token_urlsafe(48)
    assert valid_muse_secret(token), "Generated MUSE_API_TOKEN failed validation."

    fd = os.open(
        str(GENERATED_TOKEN_FILE),
        os.O_WRONLY | os.O_CREAT | os.O_TRUNC,
        0o600,
    )
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            f.write(token + "\n")
    except Exception:
        try:
            os.close(fd)
        except OSError:
            pass
        raise

    os.chmod(GENERATED_TOKEN_FILE, 0o600)
    file_mode = GENERATED_TOKEN_FILE.stat().st_mode & 0o777
    assert file_mode == 0o600, f"Generated token file mode is {oct(file_mode)}, expected 0o600."

    secret_source = "auto_generated_file"
    GENERATED_TOKEN_CREATED_THIS_RUN = True

    print("WARNING: Kaggle Secret MUSE_API_TOKEN was not configured.")
    print("A temporary secure token was generated automatically.")
    print(f"TOKEN_FILE={GENERATED_TOKEN_FILE}")
    print("TOKEN_FILE_MODE=0600")
    print("IMPORTANT: The token value is NOT printed.")
    print("IMPORTANT: Do not publish or include this secret file in Notebook Outputs.")
    print(
        "IMPORTANT: Normal cleanup deletes this file when "
        "AUTO_DELETE_GENERATED_TOKEN_AT_END=True."
    )

assert valid_muse_secret(token)

os.environ["MUSE_API_TOKEN"] = token

SESSION["secret_source"] = secret_source
SESSION["secret_value_persisted"] = (secret_source == "auto_generated_file")
# Backward-compatible field above means "written to a runtime file", not durable publication storage.
SESSION["secret_written_to_ephemeral_runtime_file"] = (secret_source == "auto_generated_file")
SESSION["secret_runtime_file_scope"] = (
    "ephemeral_runtime_only" if secret_source == "auto_generated_file" else "not_applicable"
)
SESSION["secret_publication_artifact_policy"] = "MUST_NOT_INCLUDE"
SESSION["secret_file_path"] = (
    str(GENERATED_TOKEN_FILE) if secret_source == "auto_generated_file" else None
)
SESSION["secret_file_mode"] = (
    "0600" if secret_source == "auto_generated_file" else None
)
SESSION["generated_token_cleanup_enabled"] = bool(
    secret_source == "auto_generated_file"
    and AUTO_DELETE_GENERATED_TOKEN_AT_END
)

print(f"MUSE_API_TOKEN_SOURCE={secret_source}")
print("MUSE_API_TOKEN_VALIDATION=PASS")
print("MUSE_API_TOKEN_VALUE=REDACTED")
if secret_source == "auto_generated_file":
    print("SECRET_RUNTIME_FILE_SCOPE=EPHEMERAL_RUNTIME_ONLY")
    print("SECRET_PUBLICATION_ARTIFACT_POLICY=MUST_NOT_INCLUDE")


## 6. Prepare a safe `cloudflared` binary / Chuẩn bị binary `cloudflared` an toàn

**English:** This cell only accepts the pinned Cloudflare `cloudflared` release when both its exact version and official SHA-256 match. System/cache candidates with a different digest are rejected; downloads are hash-checked before the binary is executed or installed. The exact version and digest are recorded in evidence.

**Tiếng Việt:** Cell này chỉ chấp nhận release `cloudflared` đã pin khi cả version chính xác và SHA-256 chính thức đều khớp. Binary hệ thống/cache có digest khác sẽ bị từ chối; file tải xuống được kiểm tra hash trước khi thực thi hoặc cài đặt. Version và digest thực tế được ghi vào evidence.


In [ ]:
BIN_DIR.mkdir(parents=True, exist_ok=True)

CLOUDFLARED_VERSION = "2026.10.0"
CLOUDFLARED_EXPECTED_SHA256 = "d33ff2d14475178d2012c2c56beba87389ac5ded27649519f198a7d3134a99db"
download_url = (
    "https://github.com/cloudflare/cloudflared/releases/download/"
    f"{CLOUDFLARED_VERSION}/cloudflared-linux-amd64"
)

def verify_cloudflared_candidate(candidate, label):
    """Accept a candidate only after the pinned digest and version match."""
    candidate_path = Path(candidate)
    if not candidate_path.is_file():
        return None

    try:
        digest = hashlib.sha256(candidate_path.read_bytes()).hexdigest()
    except OSError as exc:
        print(
            f"CLOUDFLARED_CANDIDATE_REJECTED label={label} "
            f"reason=read_error type={type(exc).__name__}"
        )
        return None

    # Check bytes before executing any candidate binary.
    if digest != CLOUDFLARED_EXPECTED_SHA256:
        print(
            f"CLOUDFLARED_CANDIDATE_REJECTED label={label} "
            f"reason=sha256_mismatch observed={digest} "
            f"expected={CLOUDFLARED_EXPECTED_SHA256}"
        )
        return None

    try:
        probe = run([str(candidate_path), "--version"], check=False)
    except OSError as exc:
        print(
            f"CLOUDFLARED_CANDIDATE_REJECTED label={label} "
            f"reason=version_probe_error type={type(exc).__name__}"
        )
        return None

    version_text = (probe.stdout or "") + (probe.stderr or "")
    if probe.returncode != 0 or CLOUDFLARED_VERSION not in version_text:
        print(
            f"CLOUDFLARED_CANDIDATE_REJECTED label={label} "
            f"reason=version_mismatch expected={CLOUDFLARED_VERSION}"
        )
        return None

    return {"path": str(candidate_path), "version": version_text.strip(), "sha256": digest}

system_cloudflared = shutil.which("cloudflared")
canonical_cloudflared = BIN_DIR / "cloudflared"
cloudflared = None
CLOUDFLARED_BOOTSTRAP_MODE = None

if system_cloudflared:
    system_candidate = verify_cloudflared_candidate(system_cloudflared, "system")
    if system_candidate is not None:
        cloudflared = system_candidate["path"]
        CLOUDFLARED_BOOTSTRAP_MODE = "system_pinned"

if cloudflared is None and canonical_cloudflared.is_file():
    cached_candidate = verify_cloudflared_candidate(canonical_cloudflared, "cache")
    if cached_candidate is not None:
        try:
            os.chmod(canonical_cloudflared, 0o755)
        except OSError:
            pass
        cloudflared = cached_candidate["path"]
        CLOUDFLARED_BOOTSTRAP_MODE = "reuse_cached_pinned"

if cloudflared is None:
    temp_cloudflared = BIN_DIR / (
        f".cloudflared.tmp.{os.getpid()}.{secrets.token_hex(6)}"
    )
    try:
        if temp_cloudflared.exists():
            temp_cloudflared.unlink()

        run([
            "curl", "-fL", "--retry", "3", "--connect-timeout", "20",
            "--proto", "=https", "--proto-redir", "=https",
            download_url, "-o", str(temp_cloudflared)
        ])

        # Verify SHA-256 before --version can execute the downloaded file.
        temp_candidate = verify_cloudflared_candidate(temp_cloudflared, "download")
        assert temp_candidate is not None, (
            "Downloaded cloudflared failed the pinned SHA-256/version check; "
            "refusing to execute or install it."
        )

        os.chmod(temp_cloudflared, 0o755)
        # Atomic replacement prevents partial writes to the canonical path.
        os.replace(temp_cloudflared, canonical_cloudflared)
        os.chmod(canonical_cloudflared, 0o755)

        cloudflared = str(canonical_cloudflared)
        CLOUDFLARED_BOOTSTRAP_MODE = "atomic_download_pinned"
    finally:
        if temp_cloudflared.exists():
            temp_cloudflared.unlink()

assert cloudflared is not None
assert Path(cloudflared).is_file(), "cloudflared binary is unavailable."
selected_candidate = verify_cloudflared_candidate(cloudflared, "selected_final")
assert selected_candidate is not None, (
    "Selected cloudflared no longer matches the pinned version and SHA-256."
)

os.environ["CLOUDFLARED_BIN"] = cloudflared
version_text = selected_candidate["version"]
sha = selected_candidate["sha256"]

print(f"CLOUDFLARED_BOOTSTRAP_MODE={CLOUDFLARED_BOOTSTRAP_MODE}")
print(f"CLOUDFLARED_BIN={cloudflared}")
print(f"CLOUDFLARED_EXPECTED_VERSION={CLOUDFLARED_VERSION}")
print(f"CLOUDFLARED_VERSION={version_text}")
print(f"CLOUDFLARED_EXPECTED_SHA256={CLOUDFLARED_EXPECTED_SHA256}")
print(f"CLOUDFLARED_SHA256={sha}")
print("CLOUDFLARED_SHA256_VERIFIED=PASS")

SESSION["cloudflared_bootstrap_mode"] = CLOUDFLARED_BOOTSTRAP_MODE
SESSION["cloudflared_expected_version"] = CLOUDFLARED_VERSION
SESSION["cloudflared_version"] = version_text
SESSION["cloudflared_expected_sha256"] = CLOUDFLARED_EXPECTED_SHA256
SESSION["cloudflared_sha256"] = sha
SESSION["cloudflared_sha256_verified"] = sha == CLOUDFLARED_EXPECTED_SHA256
assert SESSION["cloudflared_sha256_verified"] is True
print("CLOUDFLARED_GATE=PASS")


## 7. Build the canonical environment and run readiness/preflight / Dựng môi trường canonical và chạy readiness/preflight

**English:** This cell removes stale repository-owned environment overrides, verifies the canonical `agent` serving profile (`CONTEXT_SIZE=131072 target, with reported adaptive fallback if memory is insufficient`), enforces the context-aware output cap up to 131072 tokens, pins and hash-verifies `cloudflared` before use, and runs repository readiness/preflight in local `proxy` mode. It verifies the running gateway's effective Context/output values. Expected gates include `DEMO_REQUEST_BUDGET_GATE=PASS`, `CANONICAL_ENVIRONMENT=PASS`, `READINESS=PASS`, and `EXTERNAL_PREFLIGHT=PASS`.

**Tiếng Việt:** Cell này loại bỏ các environment override cũ thuộc repository, xác minh profile serving `agent` với Context mục tiêu 131072 và fallback thích ứng khi thiếu VRAM. Smoke request dùng 2048 token để qualification; gateway cho phép tối đa 131072 token nhưng tự giới hạn theo Context đã load thực tế. Cell kiểm tra version/SHA-256 của `cloudflared` trước khi dùng và chạy readiness/preflight ở chế độ local `proxy`. Các gate mong đợi gồm `DEMO_REQUEST_BUDGET_GATE=PASS`, `CANONICAL_ENVIRONMENT=PASS`, `READINESS=PASS` và `EXTERNAL_PREFLIGHT=PASS`.


In [ ]:
env, removed_repo_overrides = build_canonical_demo_env(
    token, cloudflared, exposure_mode="proxy"
)

assert env.get("MUSE_API_TOKEN") == token
assert env.get("CLOUDFLARED_BIN") == cloudflared

print(f"CANONICAL_ENV_REMOVED_OVERRIDE_COUNT={len(removed_repo_overrides)}")
if removed_repo_overrides:
    print("CANONICAL_ENV_REMOVED_KEYS=" + ",".join(removed_repo_overrides))

effective_config = canonical_effective_config(env)

print("CANONICAL_EFFECTIVE_CONFIG_BEGIN")
for key in sorted(effective_config):
    print(f"{key}={effective_config[key]}")
print("CANONICAL_EFFECTIVE_CONFIG_END")

assert effective_config == EXPECTED_CANONICAL_EFFECTIVE_CONFIG, (
    "Persistent serving config does not resolve to the exact pinned v1.0.0 "
    f"agent profile.\nexpected={EXPECTED_CANONICAL_EFFECTIVE_CONFIG}\n"
    f"actual={effective_config}"
)


gateway_max_tokens = gateway_policy_max_tokens(env)
minimum_visible_answer_reserve = int(effective_config["MIN_ANSWER_TOKENS"])

print(f"GATEWAY_POLICY_MAX_TOKENS={gateway_max_tokens}")
print(f"DEMO_MIN_VISIBLE_ANSWER_RESERVE={minimum_visible_answer_reserve}")
print(f"DEMO_REQUEST_MAX_TOKENS={DEMO_REQUEST_MAX_TOKENS}")
print(f"DEMO_REASONING_STRENGTH={DEMO_REASONING_STRENGTH}")

assert gateway_max_tokens == 131072, (
    f"Agent gateway max_tokens ceiling drifted: {gateway_max_tokens} != 131072"
)
assert DEMO_REQUEST_MAX_TOKENS <= gateway_max_tokens, (
    "Demo request exceeds the pinned public gateway max_tokens cap."
)
assert DEMO_REQUEST_MAX_TOKENS >= minimum_visible_answer_reserve, (
    "Demo request token budget is smaller than the canonical visible-answer reserve."
)
assert DEMO_REASONING_STRENGTH == "high"

SESSION["gateway_policy_max_tokens"] = gateway_max_tokens
SESSION["demo_min_visible_answer_reserve"] = minimum_visible_answer_reserve
SESSION["demo_request_max_tokens"] = DEMO_REQUEST_MAX_TOKENS
SESSION["demo_reasoning_strength"] = DEMO_REASONING_STRENGTH
SESSION["demo_reasoning_control"] = "chat_template_kwargs.reasoning_strength"
SESSION["demo_request_budget_gate"] = "PASS"
SESSION["core_exposure_mode"] = "proxy"

print("DEMO_REQUEST_BUDGET_GATE=PASS")
print("REQUEST_REASONING_POLICY=HIGH")
print("CORE_EXPOSURE_MODE=proxy")

SESSION["canonical_env_removed_override_count"] = len(removed_repo_overrides)
SESSION["canonical_env_removed_keys"] = removed_repo_overrides
SESSION["canonical_effective_config"] = effective_config
SESSION["canonical_environment"] = "PASS"

print("CANONICAL_ENVIRONMENT=PASS")

readiness = run(
    ["./readiness.sh", "--format", "env"],
    cwd=WORKDIR,
    env=env,
).stdout

preflight = run(
    ["./external.sh", "preflight"],
    cwd=WORKDIR,
    env=env,
).stdout

SESSION["readiness_exit"] = 0
SESSION["external_preflight_exit"] = 0
print("READINESS=PASS")
print("EXTERNAL_PREFLIGHT=PASS")

GPU_PRESTART_SNAPSHOT = query_gpu_snapshot("prestart-after-recovery")
assert len(GPU_PRESTART_SNAPSHOT) == 2
SESSION["gpu_prestart_snapshot"] = GPU_PRESTART_SNAPSHOT
print("GPU_PRESTART_SNAPSHOT_GATE=PASS")


## Start the production-style demo / Khởi động demo theo phong cách production

### English

The next cell starts **the local production core only**:

- `llama-server` loopback backend on `127.0.0.1:8088`;
- DFlash2 speculative decoding;
- Bearer-authenticated gateway on `127.0.0.1:8090`;
- **no Cloudflare Quick Tunnel dependency in this phase**.

After the local auth smoke, real non-streaming generation, and SSE generation have been qualified, a later dedicated phase attempts the optional Cloudflare Quick Tunnel transport separately.

The repository itself performs the strict source/runtime/model identity checks before serving.

If any later cell fails after startup, run the final **Cleanup** cell manually.

---

### Tiếng Việt

Cell tiếp theo khởi động **chỉ local production core**:

- `llama-server` loopback backend tại `127.0.0.1:8088`;
- speculative decoding bằng DFlash2;
- Bearer-authenticated gateway tại `127.0.0.1:8090`;
- **phase này không phụ thuộc vào Cloudflare Quick Tunnel**.

Sau khi local auth smoke, real non-streaming generation và SSE generation được qualification thành công, một phase riêng ở phía sau mới thử transport qua Cloudflare Quick Tunnel tùy chọn.

Chính repository thực hiện các kiểm tra nghiêm ngặt về source/runtime/model identity trước khi phục vụ request.

Nếu bất kỳ cell nào phía sau bị lỗi sau khi startup, hãy chạy thủ công cell **Cleanup** cuối cùng.


## 8. Start the authenticated local production core / Khởi động local production core có xác thực

**English:** This cell starts the backend and authenticated loopback gateway with `EXPOSURE_MODE=proxy`. It verifies that the live gateway process uses the exact Context/output budget loaded by the backend, then re-checks the fork source manifest. Configured gateway limits are bounded (idle timeout 180 seconds, hard deadline 3600 seconds); the gateway clamps generation to the effective runtime budget. No Quick Tunnel is required in this phase. Success includes `GATEWAY_RUNTIME_LIMITS_PROCESS_GATE=PASS`, `POST_START_SOURCE_MANIFEST=PASS`, `CORE_PROXY_START=PASS`, and local endpoint `http://127.0.0.1:8090`.

**Tiếng Việt:** Cell này khởi động backend và gateway loopback có xác thực với `EXPOSURE_MODE=proxy`. Cell xác minh tiến trình gateway đang chạy dùng đúng Context/output budget mà backend đã load, rồi kiểm tra lại source manifest của fork. Timeout gateway được giới hạn (idle 180 giây, hard deadline 3600 giây); gateway tự giới hạn output theo runtime budget thực tế. Giai đoạn này không cần Quick Tunnel. Thành công gồm `GATEWAY_RUNTIME_LIMITS_PROCESS_GATE=PASS`, `POST_START_SOURCE_MANIFEST=PASS`, `CORE_PROXY_START=PASS` và local endpoint `http://127.0.0.1:8090`.


In [ ]:
# Core phase: start only the authenticated local proxy. Public transport is
# qualified later, after real model generation/SSE.
env, _removed_at_start = build_canonical_demo_env(
    token,
    cloudflared,
    exposure_mode="proxy",
)

assert "token" in globals(), (
    "Secret setup cell has not run. Run the MUSE_API_TOKEN cell before starting."
)
assert env.get("MUSE_API_TOKEN") == token
assert env.get("CLOUDFLARED_BIN")

start_result = run(
    ["./external.sh", "start"],
    cwd=WORKDIR,
    env=env,
).stdout

# Trust only exact verification records written by the HF-only startup path.
import shlex

def read_state(path):
    values = {}
    for raw in Path(path).read_text(encoding="utf-8").splitlines():
        if not raw or "=" not in raw:
            continue
        key, val = raw.split("=", 1)
        parts = shlex.split(val)
        values[key] = parts[0] if parts else ""
    return values

state_dir = WORKDIR / "artifacts" / "runtime-state"
target_state = read_state(state_dir / "model.env")
draft_state = read_state(state_dir / "dflash.env")
server_metadata = json.loads((state_dir / "server-metadata.json").read_text(encoding="utf-8"))

assert target_state.get("MODEL_RUNTIME_SOURCE") in {"huggingface_download", "hf_verified_cache"}, target_state
assert target_state.get("MODEL_HF_REPO") == TARGET_HF_REPO, target_state
assert target_state.get("MODEL_HF_REVISION") == TARGET_HF_REVISION, target_state
assert int(target_state.get("MODEL_RESOLVED_SIZE_BYTES", "-1")) == TARGET_EXPECTED_SIZE_BYTES, target_state
assert target_state.get("MODEL_RESOLVED_SHA256") == TARGET_EXPECTED_SHA256, target_state

assert draft_state.get("DFLASH_HF_REPO") == DRAFT_HF_REPO, draft_state
assert draft_state.get("DFLASH_HF_REVISION") == DRAFT_HF_REVISION, draft_state
assert int(draft_state.get("DFLASH_RESOLVED_SIZE_BYTES", "-1")) == DRAFT_EXPECTED_SIZE_BYTES, draft_state
assert draft_state.get("DFLASH_RESOLVED_SHA256") == DRAFT_EXPECTED_SHA256, draft_state

assert server_metadata.get("mmproj_path") == str(MM_PROJ_PATH), server_metadata
assert int(server_metadata.get("mmproj_size_bytes", -1)) == MM_PROJ_EXPECTED_SIZE_BYTES, server_metadata
assert server_metadata.get("mmproj_sha256") == MM_PROJ_EXPECTED_SHA256, server_metadata
assert 4096 <= int(server_metadata.get("context_size", 0)) <= 131072, server_metadata
assert 1 <= int(server_metadata.get("max_tokens", 0)) <= 131072, server_metadata

SESSION["target_observed_sha256"] = target_state["MODEL_RESOLVED_SHA256"]
SESSION["draft_observed_sha256"] = draft_state["DFLASH_RESOLVED_SHA256"]
SESSION["mmproj_observed_sha256"] = server_metadata["mmproj_sha256"]
SESSION["target_sha256_seconds"] = float(target_state.get("MODEL_VERIFICATION_SECONDS", "0") or 0)
SESSION["draft_sha256_seconds"] = None
SESSION["effective_context_size"] = int(server_metadata["context_size"])
SESSION["effective_max_output_tokens"] = min(int(server_metadata["context_size"]), int(server_metadata["max_tokens"]))
SESSION["model_sha256_gate"] = "PASS"
SESSION["target_match_count"] = 1
SESSION["draft_match_count"] = 1
SESSION["prebuilt_runtime_root_count"] = 0
SESSION["runtime_roots"] = []
SESSION["target_variation"] = "hf-pinned"
SESSION["draft_variation"] = "hf-pinned"
SESSION["target_variation_gate"] = "PASS"
SESSION["draft_variation_gate"] = "PASS"
print(f"HF_TARGET_SHA256={SESSION['target_observed_sha256']}")
print(f"HF_DFLASH_SHA256={SESSION['draft_observed_sha256']}")
print(f"HF_MMPROJ_SHA256={SESSION['mmproj_observed_sha256']}")
print(f"SERVER_EFFECTIVE_CONTEXT={SESSION['effective_context_size']}")
print(f"SERVER_EFFECTIVE_MAX_OUTPUT_TOKENS={SESSION['effective_max_output_tokens']}")
print("HF_MODEL_INTEGRITY_GATE=PASS")

auth_proxy_pid = verify_auth_proxy_runtime_limits(
    SESSION["effective_context_size"], SESSION["effective_max_output_tokens"]
)
SESSION["auth_proxy_pid_verified"] = True
SESSION["gateway_runtime_limits_process_gate"] = "PASS"

post_start_manifest = run(
    ["python3", "-B", "scripts/source_manifest.py", "check", "--root", "."],
    cwd=WORKDIR,
    env=env,
).stdout
assert "SOURCE_MANIFEST: OK" in post_start_manifest
SESSION["post_start_source_manifest"] = "PASS"
print("POST_START_SOURCE_MANIFEST=PASS")

status_env = run(
    ["./external.sh", "status", "--format", "env"],
    cwd=WORKDIR,
    env=env,
).stdout
status_map = parse_env_blob(status_env)

assert status_map.get("EXPOSURE_STATUS") in {"proxy-ready", "ready"}, status_map
assert status_map.get("MODE") == "proxy", status_map
assert status_map.get("PROXY_OWNED") == "1", status_map
assert status_map.get("TUNNEL_OWNED") == "0", status_map

LOCAL_URL = run(
    ["./external.sh", "local-endpoint"],
    cwd=WORKDIR,
    env=env,
).stdout.strip().rstrip("/")

assert LOCAL_URL == "http://127.0.0.1:8090", LOCAL_URL

PUBLIC_URL = None

SESSION["start_exit"] = 0
SESSION["status_env_captured"] = True
SESSION["local_gateway_url"] = LOCAL_URL
SESSION["core_exposure_mode"] = "proxy"
SESSION["core_proxy_status"] = status_map.get("EXPOSURE_STATUS")
SESSION["backend_provenance"] = status_map.get("BACKEND_PROVENANCE")
SESSION["backend_status"] = status_map.get("BACKEND_STATUS")
SESSION["backend_healthy"] = status_map.get("BACKEND_HEALTHY")
SESSION["runtime_status_summary"] = {
    key: status_map.get(key)
    for key in (
        "VERSION", "MODE", "AUTH_ENABLED", "MODEL", "GPU_COUNT",
        "BACKEND_STATUS", "BACKEND_HEALTHY", "BACKEND_PROVENANCE",
        "PROXY_STATUS", "TUNNEL_STATUS", "EXPOSURE_STATUS",
    )
}

print(f"LOCAL_GATEWAY={LOCAL_URL}")
print("CORE_PROXY_START=PASS")
print("PRODUCTION_DEMO_START=PASS")

VENV_FALLBACK_WARNING = (
    "WARNING: standard venv bootstrap could not complete; using ensurepip-less fallback."
)
VENV_FALLBACK_MARKER = "VENV_ENSUREPIP_FALLBACK=USED"

venv_fallback_warning_seen = VENV_FALLBACK_WARNING in start_result
venv_fallback_marker_seen = VENV_FALLBACK_MARKER in start_result
venv_raw_ensurepip_error_seen = (
    "Error: Command" in start_result
    and "ensurepip" in start_result
)

assert venv_fallback_warning_seen == venv_fallback_marker_seen, (
    "Kaggle venv fallback observability mismatch: the human-readable warning "
    "and machine-readable marker must appear together."
)
assert not venv_raw_ensurepip_error_seen, (
    "Expected ensurepip failure leaked as a top-level raw Error: Command line."
)

SESSION["venv_fallback_observed"] = bool(venv_fallback_marker_seen)
SESSION["venv_raw_ensurepip_error_suppressed"] = True
SESSION["venv_log_polish_gate"] = "PASS"

print(
    "VENV_ENSUREPIP_FALLBACK_OBSERVED="
    + ("YES" if venv_fallback_marker_seen else "NO")
)
print("VENV_RAW_ENSUREPIP_ERROR_SUPPRESSED=PASS")
print("VENV_LOG_POLISH_GATE=PASS")

if "ensurepip-less fallback" in start_result:
    SESSION["venv_bootstrap_mode"] = "validated_ensurepip_less_fallback"
    print("VENV_BOOTSTRAP_MODE=validated_ensurepip_less_fallback")
    print("VENV_BOOTSTRAP_GATE=PASS")
else:
    SESSION["venv_bootstrap_mode"] = "standard_or_reused"
    print("VENV_BOOTSTRAP_MODE=standard_or_reused")
    print("VENV_BOOTSTRAP_GATE=PASS")

assert SESSION.get("backend_provenance") == "PASS", status_map
print("MODEL_RUNTIME_PROVENANCE=PASS")


## 9. Verify the authenticated local API boundary / Xác minh biên API local có xác thực

**English:** This cell checks that unauthenticated `/v1/models` is rejected with HTTP 401, while authenticated `/ready` and `/v1/models` return HTTP 200 through `127.0.0.1:8090`. This isolates auth/API correctness from public DNS and tunnel behavior. Success is `LOCAL_AUTH_GATEWAY=PASS`.

**Tiếng Việt:** Cell này kiểm tra `/v1/models` không có Bearer token phải bị từ chối với HTTP 401, trong khi `/ready` và `/v1/models` có xác thực phải trả HTTP 200 qua `127.0.0.1:8090`. Việc này tách correctness của auth/API khỏi DNS và tunnel công khai. Thành công là `LOCAL_AUTH_GATEWAY=PASS`.


In [ ]:
# Mandatory core smoke goes through the authenticated loopback gateway.
# This isolates model/runtime/API correctness from external DNS/tunnel transport.
import requests

AUTH = {"Authorization": f"Bearer {token}"}

unauth_models = requests.get(
    f"{LOCAL_URL}/v1/models",
    timeout=30,
)
print("LOCAL_UNAUTH_V1_MODELS_HTTP=", unauth_models.status_code)
assert unauth_models.status_code == 401, (
    f"Expected 401 from local protected /v1/models without Bearer token, "
    f"got {unauth_models.status_code}"
)

ready = requests.get(
    f"{LOCAL_URL}/ready",
    headers=AUTH,
    timeout=60,
)
print("LOCAL_AUTH_READY_HTTP=", ready.status_code)
print("LOCAL_AUTH_READY_BODY=", ready.text.strip()[:500])
assert ready.status_code == 200, ready.text

models = requests.get(
    f"{LOCAL_URL}/v1/models",
    headers=AUTH,
    timeout=60,
)
print("LOCAL_AUTH_V1_MODELS_HTTP=", models.status_code)
assert models.status_code == 200, models.text
models_json = models.json()
print(json.dumps(models_json, indent=2)[:2000])

SESSION["local_unauth_v1_models_http"] = unauth_models.status_code
SESSION["local_auth_ready_http"] = ready.status_code
SESSION["local_auth_v1_models_http"] = models.status_code

print("LOCAL_AUTH_GATEWAY=PASS")


## 10. Verify dual-T4 runtime residency / Xác minh runtime residency trên cả hai T4

**English.** GPU topology alone does not prove that the loaded runtime resides on both devices. This gate compares a clean pre-start snapshot (captured after stale-process recovery) with the post-load snapshot and requires a material VRAM increase on **each** T4. This is a residency/provenance check, not a performance benchmark and not a claim that instantaneous utilization must be non-zero at the sampling moment.

**Tiếng Việt.** Chỉ thấy topology GPU chưa đủ chứng minh runtime đã nằm trên cả hai thiết bị. Gate này so sánh snapshot sạch trước start (sau stale-process recovery) với snapshot sau khi model load và yêu cầu VRAM tăng đáng kể trên **từng** T4. Đây là kiểm chứng residency/provenance, không phải benchmark hiệu năng và không yêu cầu utilization tức thời phải khác 0 đúng tại thời điểm lấy mẫu.


In [ ]:
GPU_POSTLOAD_SNAPSHOT = query_gpu_snapshot("postload-ready")
assert len(GPU_PRESTART_SNAPSHOT) == 2
assert len(GPU_POSTLOAD_SNAPSHOT) == 2

pre_by_index = {row["index"]: row for row in GPU_PRESTART_SNAPSHOT}
post_by_index = {row["index"]: row for row in GPU_POSTLOAD_SNAPSHOT}
assert set(pre_by_index) == {0, 1}
assert set(post_by_index) == {0, 1}

GPU_RESIDENCY_DELTAS_MIB = {}
for index in (0, 1):
    delta = (
        post_by_index[index]["memory_used_mib"]
        - pre_by_index[index]["memory_used_mib"]
    )
    GPU_RESIDENCY_DELTAS_MIB[index] = delta
    print(f"GPU_{index}_VRAM_DELTA_MIB={delta}")
    if delta < DUAL_GPU_MIN_RESIDENCY_DELTA_MIB:
        capture_backend_forensics("dual-gpu-residency-failure", env=env)
        raise AssertionError(
            f"GPU {index} residency delta {delta} MiB is below "
            f"{DUAL_GPU_MIN_RESIDENCY_DELTA_MIB} MiB."
        )

SESSION["gpu_postload_snapshot"] = GPU_POSTLOAD_SNAPSHOT
SESSION["gpu_residency_deltas_mib"] = {
    str(k): v for k, v in GPU_RESIDENCY_DELTAS_MIB.items()
}
SESSION["dual_gpu_residency_min_delta_mib"] = DUAL_GPU_MIN_RESIDENCY_DELTA_MIB
SESSION["dual_gpu_residency_gate"] = "PASS"
print("DUAL_GPU_MEMORY_RESIDENCY=PASS")


## 11. Run one real non-streaming Muse generation with full observable transcript / Chạy một real Muse generation non-stream với transcript quan sát đầy đủ

**English:** This is the first real production semantic test. It sends the configured qualification request budget (2048 tokens) with `chat_template_kwargs.reasoning_strength=high`; the server/gateway ceiling remains 131072 tokens or the actually loaded Context, whichever is lower. For reviewer-visible evidence, the cell prints a complete **sanitized request transcript** (method, local URL, headers with the Bearer credential redacted, and full JSON body), then the HTTP status, response headers, exact raw response body, parsed JSON response, emitted reasoning field when present, visible assistant content, usage, and `finish_reason`. The prompt explicitly asks the model to describe only facts directly demonstrated by this notebook and forbids unsupported architecture/security claims. Success is `LOCAL_NONSTREAM_DEMO=PASS`. On HTTP or semantic failure, the same transcript is preserved before bounded secret-free forensics are captured.

**Tiếng Việt:** Đây là bài test semantic thực tế đầu tiên của production notebook. Request gửi budget qualification đã cấu hình (2048 token) cùng `chat_template_kwargs.reasoning_strength=high`; trần server/gateway vẫn là 131072 token hoặc Context được load thực tế nếu thấp hơn. Để tạo evidence mà reviewer có thể kiểm tra trực tiếp, cell in **sanitized request transcript đầy đủ** (method, local URL, headers với Bearer credential đã được che và toàn bộ JSON body), sau đó in HTTP status, response headers, raw response body chính xác, parsed JSON response, reasoning field do backend phát ra nếu có, visible assistant content, usage và `finish_reason`. Prompt yêu cầu model chỉ mô tả những facts được notebook trực tiếp chứng minh và cấm các claim kiến trúc/bảo mật chưa được xác minh. Thành công là `LOCAL_NONSTREAM_DEMO=PASS`. Nếu HTTP hoặc semantic FAIL, cùng transcript đó vẫn được lưu trước khi notebook thu thập forensic có giới hạn và không chứa secret.


In [ ]:
LOCAL_NONSTREAM_FINISH_REASON = None
LOCAL_NONSTREAM_STATUS = None
LOCAL_NONSTREAM_FAILURE_CAPTURED = False
LOCAL_NONSTREAM_ERROR_CODE = None
LOCAL_NONSTREAM_ERROR_TYPE = None
LOCAL_NONSTREAM_REQUEST_ID = None
LOCAL_NONSTREAM_SEMANTIC_FAILURE = None
LOCAL_NONSTREAM_CONTENT_CHARS = 0
LOCAL_NONSTREAM_REASONING_CHARS = 0
LOCAL_NONSTREAM_COMPLETION_TOKENS = None
LOCAL_NONSTREAM_PROMPT_TOKENS = None
LOCAL_NONSTREAM_DRAFT_N = None
LOCAL_NONSTREAM_DRAFT_N_ACCEPTED = None
LOCAL_NONSTREAM_PREDICTED_PER_SECOND = None

INTERACTION_EVIDENCE = {
    "schema": "muse-glimmer-30b.interaction-evidence.v2",
    "release": EXPECTED_VERSION,
    "source_ref": SOURCE_REF,
    "source_head": SESSION.get("actual_head"),
    "secret_policy": {
        "authorization_header": "redacted",
        "bearer_token_written": False,
    },
    "nonstream": None,
    "sse": None,
    "semantic_sanity": None,
}

def redact_http_headers(headers):
    sanitized = {}
    for key, value in dict(headers).items():
        if str(key).lower() == "authorization":
            sanitized[str(key)] = "Bearer <REDACTED>"
        else:
            sanitized[str(key)] = str(value)
    return sanitized

def write_interaction_evidence():
    serialized = json.dumps(
        INTERACTION_EVIDENCE,
        indent=2,
        ensure_ascii=False,
    ) + "\n"
    assert token not in serialized, "Bearer credential leaked into interaction evidence."
    INTERACTION_EVIDENCE_PATH.write_text(serialized, encoding="utf-8")
    return serialized

if INTERACTION_EVIDENCE_PATH.exists():
    INTERACTION_EVIDENCE_PATH.unlink()

if RUN_NONSTREAM_DEMO:
    payload = {
        "model": "muse-glimmer-30B",
        "messages": [{
            "role": "user",
            "content": (
                "In exactly two concise sentences, describe only what this notebook "
                "directly verifies: Muse-Glimmer-30B runs on Kaggle NVIDIA T4 x2, "
                "serves through the Bearer-authenticated local gateway, uses the "
                "configured DFlash2 speculative-decoding path, and returns visible "
                "assistant output. Do not claim mTLS, service mesh, HA, rate limiting, "
                "prompt filtering, or any control not explicitly demonstrated here."
            ),
        }],
        "max_tokens": DEMO_REQUEST_MAX_TOKENS,
        "chat_template_kwargs": {"reasoning_strength": DEMO_REASONING_STRENGTH},
    }

    request_url = f"{LOCAL_URL}/v1/chat/completions"
    request_headers = {**AUTH, "Content-Type": "application/json"}
    request_evidence = {
        "method": "POST",
        "url": request_url,
        "headers": redact_http_headers(request_headers),
        "json": payload,
    }

    print("\n=== LOCAL NON-STREAM REQUEST (SANITIZED) ===")
    print(json.dumps(request_evidence, indent=2, ensure_ascii=False))
    print("=== END LOCAL NON-STREAM REQUEST ===")

    request_started = time.monotonic()
    r = requests.post(
        request_url,
        headers=request_headers,
        json=payload,
        timeout=3600,
    )
    request_elapsed = time.monotonic() - request_started

    LOCAL_NONSTREAM_STATUS = r.status_code
    response_headers = {str(k): str(v) for k, v in r.headers.items()}
    raw_response_body = r.text
    try:
        parsed_response_body = r.json()
    except Exception:
        parsed_response_body = None

    response_evidence = {
        "status_code": r.status_code,
        "headers": response_headers,
        "raw_body": raw_response_body,
        "json": parsed_response_body,
        "elapsed_seconds": round(request_elapsed, 6),
    }
    INTERACTION_EVIDENCE["nonstream"] = {
        "request": request_evidence,
        "response": response_evidence,
    }
    write_interaction_evidence()

    print("\n=== LOCAL NON-STREAM RESPONSE ===")
    print(f"HTTP_STATUS={r.status_code}")
    print(f"ELAPSED_SECONDS={request_elapsed:.3f}")
    print("--- response headers ---")
    print(json.dumps(response_headers, indent=2, ensure_ascii=False))
    print("--- raw response body ---")
    print(raw_response_body)
    print("--- parsed response JSON ---")
    if parsed_response_body is None:
        print("<NON_JSON_RESPONSE>")
    else:
        print(json.dumps(parsed_response_body, indent=2, ensure_ascii=False))
    print("=== END LOCAL NON-STREAM RESPONSE ===")

    print("LOCAL_NONSTREAM_HTTP=", r.status_code)
    print(f"LOCAL_NONSTREAM_ELAPSED_SECONDS={request_elapsed:.3f}")
    print(f"LOCAL_NONSTREAM_REQUEST_MAX_TOKENS={DEMO_REQUEST_MAX_TOKENS}")
    print(f"LOCAL_NONSTREAM_REASONING_STRENGTH={DEMO_REASONING_STRENGTH}")

    if r.status_code != 200:
        LOCAL_NONSTREAM_FAILURE_CAPTURED = True

        try:
            error_payload = parsed_response_body or {}
            error_obj = error_payload.get("error") or {}
            LOCAL_NONSTREAM_ERROR_CODE = error_obj.get("code")
            LOCAL_NONSTREAM_ERROR_TYPE = error_obj.get("type")
            LOCAL_NONSTREAM_REQUEST_ID = (
                error_payload.get("request_id")
                or error_obj.get("request_id")
            )
        except Exception:
            pass

        print(f"LOCAL_NONSTREAM_ERROR_CODE={LOCAL_NONSTREAM_ERROR_CODE}")
        print(f"LOCAL_NONSTREAM_ERROR_TYPE={LOCAL_NONSTREAM_ERROR_TYPE}")
        print(f"LOCAL_NONSTREAM_REQUEST_ID={LOCAL_NONSTREAM_REQUEST_ID}")

        forensic_summary = capture_backend_forensics(
            "local-nonstream-http-failure",
            response=r,
            env=env,
        )

        SESSION["local_nonstream_forensic_capture"] = "PASS"
        SESSION["local_nonstream_forensic_summary"] = forensic_summary
        print("LOCAL_NONSTREAM_DEMO=FAIL_CAPTURED")
    else:
        data = parsed_response_body
        assert isinstance(data, dict), "HTTP 200 response must be a JSON object."

        choices = data.get("choices") or []
        choice = choices[0] if choices else {}
        message = choice.get("message") or {}
        usage = data.get("usage") or {}
        timings = data.get("timings") or {}

        LOCAL_NONSTREAM_FINISH_REASON = choice.get("finish_reason")
        content = message.get("content") or ""

        reasoning_text = (
            message.get("reasoning_content")
            or message.get("reasoning")
            or ""
        )
        if not isinstance(reasoning_text, str):
            reasoning_text = str(reasoning_text)

        LOCAL_NONSTREAM_CONTENT_CHARS = len(content)
        LOCAL_NONSTREAM_REASONING_CHARS = len(reasoning_text)
        LOCAL_NONSTREAM_COMPLETION_TOKENS = usage.get("completion_tokens")
        LOCAL_NONSTREAM_PROMPT_TOKENS = usage.get("prompt_tokens")
        LOCAL_NONSTREAM_DRAFT_N = timings.get("draft_n")
        LOCAL_NONSTREAM_DRAFT_N_ACCEPTED = timings.get("draft_n_accepted")
        LOCAL_NONSTREAM_PREDICTED_PER_SECOND = timings.get("predicted_per_second")

        print("\n--- reasoning emitted by backend (non-stream) ---")
        print(reasoning_text if reasoning_text else "<EMPTY>")
        print("--- end reasoning ---")
        print("\n--- assistant (local gateway) ---")
        print(content)
        print("--- end assistant ---")
        print("finish_reason=", LOCAL_NONSTREAM_FINISH_REASON)
        print("visible_content_chars=", LOCAL_NONSTREAM_CONTENT_CHARS)
        print("reasoning_chars=", LOCAL_NONSTREAM_REASONING_CHARS)
        print("prompt_tokens=", LOCAL_NONSTREAM_PROMPT_TOKENS)
        print("completion_tokens=", LOCAL_NONSTREAM_COMPLETION_TOKENS)
        print("usage=", json.dumps(usage, ensure_ascii=False, sort_keys=True))
        print("timings=", json.dumps(timings, ensure_ascii=False, sort_keys=True))
        print("draft_n=", LOCAL_NONSTREAM_DRAFT_N)
        print("draft_n_accepted=", LOCAL_NONSTREAM_DRAFT_N_ACCEPTED)
        print("predicted_per_second=", LOCAL_NONSTREAM_PREDICTED_PER_SECOND)

        if not content.strip():
            LOCAL_NONSTREAM_SEMANTIC_FAILURE = "EMPTY_VISIBLE_CONTENT"
        elif LOCAL_NONSTREAM_FINISH_REASON not in {"stop", "length"}:
            LOCAL_NONSTREAM_SEMANTIC_FAILURE = "INVALID_FINISH_REASON"

        if LOCAL_NONSTREAM_SEMANTIC_FAILURE:
            LOCAL_NONSTREAM_FAILURE_CAPTURED = True
            print(
                "LOCAL_NONSTREAM_SEMANTIC_FAILURE="
                + LOCAL_NONSTREAM_SEMANTIC_FAILURE
            )
            forensic_summary = capture_backend_forensics(
                "local-nonstream-semantic-failure",
                response=r,
                env=env,
            )
            SESSION["local_nonstream_forensic_capture"] = "PASS"
            SESSION["local_nonstream_forensic_summary"] = forensic_summary
            print("LOCAL_NONSTREAM_DEMO=FAIL_CAPTURED")
        else:
            print("LOCAL_NONSTREAM_DEMO=PASS")
else:
    INTERACTION_EVIDENCE["nonstream"] = {"status": "SKIPPED"}
    write_interaction_evidence()
    print("LOCAL_NONSTREAM_DEMO=SKIPPED")

SESSION["local_nonstream_http"] = LOCAL_NONSTREAM_STATUS
SESSION["local_nonstream_finish_reason"] = LOCAL_NONSTREAM_FINISH_REASON
SESSION["local_nonstream_error_code"] = LOCAL_NONSTREAM_ERROR_CODE
SESSION["local_nonstream_error_type"] = LOCAL_NONSTREAM_ERROR_TYPE
SESSION["local_nonstream_request_id"] = LOCAL_NONSTREAM_REQUEST_ID
SESSION["local_nonstream_failure_captured"] = LOCAL_NONSTREAM_FAILURE_CAPTURED
SESSION["local_nonstream_semantic_failure"] = LOCAL_NONSTREAM_SEMANTIC_FAILURE
SESSION["local_nonstream_content_chars"] = LOCAL_NONSTREAM_CONTENT_CHARS
SESSION["local_nonstream_reasoning_chars"] = LOCAL_NONSTREAM_REASONING_CHARS
SESSION["local_nonstream_completion_tokens"] = LOCAL_NONSTREAM_COMPLETION_TOKENS
SESSION["local_nonstream_prompt_tokens"] = LOCAL_NONSTREAM_PROMPT_TOKENS
SESSION["local_nonstream_draft_n"] = LOCAL_NONSTREAM_DRAFT_N
SESSION["local_nonstream_draft_n_accepted"] = LOCAL_NONSTREAM_DRAFT_N_ACCEPTED
SESSION["local_nonstream_predicted_per_second"] = LOCAL_NONSTREAM_PREDICTED_PER_SECOND

SESSION["local_nonstream_reasoning_strength"] = DEMO_REASONING_STRENGTH
SESSION["local_nonstream_visible_answer"] = (
    LOCAL_NONSTREAM_STATUS == 200
    and not LOCAL_NONSTREAM_FAILURE_CAPTURED
    and LOCAL_NONSTREAM_CONTENT_CHARS > 0
)


## 12. Run one real SSE generation with full protocol transcript / Chạy một real SSE generation với protocol transcript đầy đủ

**English:** This cell runs the second real production model qualification through SSE, again using `reasoning_strength=high`. It prints the complete sanitized request, HTTP response headers, **every decoded SSE protocol line in order (including blank frame separators and the terminal `data: [DONE]`)**, and then reconstructs both visible assistant content and any reasoning field emitted by the backend. The raw event sequence is also written to the interaction-evidence artifact. Acceptance requires HTTP 200, non-empty visible streamed content, a terminal `finish_reason` of `stop` or `length`, and the `[DONE]` sentinel. If the non-stream test failed, this cell intentionally skips SSE. Success is `LOCAL_SSE_DEMO=PASS`.

**Tiếng Việt:** Cell này chạy qualification thực tế thứ hai của production notebook qua SSE, tiếp tục dùng `reasoning_strength=high`. Cell in toàn bộ sanitized request, HTTP response headers, **mọi dòng protocol SSE đã decode theo đúng thứ tự (kể cả blank separator giữa frame và sentinel cuối `data: [DONE]`)**, rồi tái dựng cả visible assistant content lẫn reasoning field mà backend phát ra nếu có. Chuỗi raw event cũng được ghi vào interaction-evidence artifact. Điều kiện đạt gồm HTTP 200, visible streamed content khác rỗng, `finish_reason` cuối là `stop` hoặc `length`, và phải thấy sentinel `[DONE]`. Nếu non-stream test đã FAIL thì cell chủ động bỏ qua SSE. Thành công là `LOCAL_SSE_DEMO=PASS`.


In [ ]:
LOCAL_SSE_STATUS = None
LOCAL_SSE_DONE = False
LOCAL_SSE_FINISH_REASON = None
LOCAL_SSE_CONTENT_CHARS = 0
LOCAL_SSE_REASONING_CHARS = 0
LOCAL_SSE_FAILURE_CAPTURED = False
LOCAL_SSE_SEMANTIC_FAILURE = None
LOCAL_SSE_RECONSTRUCTED_CONTENT = ""
LOCAL_SSE_RECONSTRUCTED_REASONING = ""
LOCAL_SSE_TIMINGS = {}
LOCAL_SSE_DRAFT_N = None
LOCAL_SSE_DRAFT_N_ACCEPTED = None
LOCAL_SSE_PREDICTED_PER_SECOND = None

if (
    RUN_SSE_DEMO
    and LOCAL_NONSTREAM_STATUS == 200
    and not LOCAL_NONSTREAM_FAILURE_CAPTURED
):
    payload = {
        "model": "muse-glimmer-30B",
        "messages": [{
            "role": "user",
            "content": (
                "In exactly two concise sentences, describe only what this notebook "
                "directly demonstrates during streaming: DFlash2 is used by the "
                "configured Muse-Glimmer-30B backend, and the Bearer-authenticated "
                "gateway mediates the API path tested by this notebook. Do not claim "
                "mTLS, service mesh, HA, rate limiting, prompt filtering, or any "
                "control not explicitly demonstrated here."
            ),
        }],
        "max_tokens": DEMO_REQUEST_MAX_TOKENS,
        "chat_template_kwargs": {"reasoning_strength": DEMO_REASONING_STRENGTH},
        "stream": True,
    }

    request_url = f"{LOCAL_URL}/v1/chat/completions"
    request_headers = {**AUTH, "Content-Type": "application/json"}
    request_evidence = {
        "method": "POST",
        "url": request_url,
        "headers": redact_http_headers(request_headers),
        "json": payload,
    }

    print("\n=== LOCAL SSE REQUEST (SANITIZED) ===")
    print(json.dumps(request_evidence, indent=2, ensure_ascii=False))
    print("=== END LOCAL SSE REQUEST ===")

    sse_raw_lines = []
    sse_parsed_events = []
    sse_content_parts = []
    sse_reasoning_parts = []
    response_headers = {}

    with requests.post(
        request_url,
        headers=request_headers,
        json=payload,
        stream=True,
        timeout=3600,
    ) as r:
        LOCAL_SSE_STATUS = r.status_code
        response_headers = {str(k): str(v) for k, v in r.headers.items()}

        print("\n=== LOCAL SSE RESPONSE HEADERS ===")
        print(f"HTTP_STATUS={r.status_code}")
        print(json.dumps(response_headers, indent=2, ensure_ascii=False))
        print("=== END LOCAL SSE RESPONSE HEADERS ===")

        print("LOCAL_SSE_HTTP=", r.status_code)
        print(f"LOCAL_SSE_REQUEST_MAX_TOKENS={DEMO_REQUEST_MAX_TOKENS}")
        print(f"LOCAL_SSE_REASONING_STRENGTH={DEMO_REASONING_STRENGTH}")

        if r.status_code != 200:
            LOCAL_SSE_FAILURE_CAPTURED = True
            raw_error_body = r.text
            print("\n=== LOCAL SSE ERROR BODY ===")
            print(raw_error_body)
            print("=== END LOCAL SSE ERROR BODY ===")

            INTERACTION_EVIDENCE["sse"] = {
                "request": request_evidence,
                "response": {
                    "status_code": r.status_code,
                    "headers": response_headers,
                    "raw_body": raw_error_body,
                },
            }
            write_interaction_evidence()

            capture_backend_forensics(
                "local-sse-http-failure",
                response=r,
                env=env,
            )
            print("LOCAL_SSE_DEMO=FAIL_CAPTURED")
        else:
            print("\n=== RAW SSE PROTOCOL TRANSCRIPT ===")
            for line in r.iter_lines(decode_unicode=True):
                if line is None:
                    line = ""
                if not isinstance(line, str):
                    line = line.decode("utf-8", errors="replace")

                sse_raw_lines.append(line)
                print(line)

                if line == "data: [DONE]":
                    LOCAL_SSE_DONE = True
                    continue
                if not line.startswith("data: "):
                    continue

                event_raw_json = line[len("data: "):]
                event = json.loads(event_raw_json)
                sse_parsed_events.append(event)

                if isinstance(event.get("timings"), dict):
                    LOCAL_SSE_TIMINGS = event["timings"]

                choices = event.get("choices") or []
                if not choices:
                    continue

                choice = choices[0]
                delta = choice.get("delta") or {}

                piece = delta.get("content") or ""
                if piece:
                    if not isinstance(piece, str):
                        piece = str(piece)
                    sse_content_parts.append(piece)
                    LOCAL_SSE_CONTENT_CHARS += len(piece)

                reasoning_piece = (
                    delta.get("reasoning_content")
                    or delta.get("reasoning")
                    or ""
                )
                if reasoning_piece:
                    if not isinstance(reasoning_piece, str):
                        reasoning_piece = str(reasoning_piece)
                    sse_reasoning_parts.append(reasoning_piece)
                    LOCAL_SSE_REASONING_CHARS += len(reasoning_piece)

                if choice.get("finish_reason") is not None:
                    LOCAL_SSE_FINISH_REASON = choice["finish_reason"]

            print("=== END RAW SSE PROTOCOL TRANSCRIPT ===")

    if LOCAL_SSE_STATUS == 200:
        LOCAL_SSE_RECONSTRUCTED_CONTENT = "".join(sse_content_parts)
        LOCAL_SSE_RECONSTRUCTED_REASONING = "".join(sse_reasoning_parts)
        LOCAL_SSE_DRAFT_N = LOCAL_SSE_TIMINGS.get("draft_n")
        LOCAL_SSE_DRAFT_N_ACCEPTED = LOCAL_SSE_TIMINGS.get("draft_n_accepted")
        LOCAL_SSE_PREDICTED_PER_SECOND = LOCAL_SSE_TIMINGS.get("predicted_per_second")

        INTERACTION_EVIDENCE["sse"] = {
            "request": request_evidence,
            "response": {
                "status_code": LOCAL_SSE_STATUS,
                "headers": response_headers,
                "raw_sse_lines": sse_raw_lines,
                "parsed_events": sse_parsed_events,
                "done_seen": LOCAL_SSE_DONE,
                "finish_reason": LOCAL_SSE_FINISH_REASON,
                "terminal_timings": LOCAL_SSE_TIMINGS,
                "reconstructed": {
                    "reasoning": LOCAL_SSE_RECONSTRUCTED_REASONING,
                    "content": LOCAL_SSE_RECONSTRUCTED_CONTENT,
                },
            },
        }
        write_interaction_evidence()

        print("\n--- reconstructed reasoning emitted by backend (SSE) ---")
        print(
            LOCAL_SSE_RECONSTRUCTED_REASONING
            if LOCAL_SSE_RECONSTRUCTED_REASONING
            else "<EMPTY>"
        )
        print("--- end reconstructed reasoning ---")
        print("\n--- reconstructed assistant stream (local gateway) ---")
        print(LOCAL_SSE_RECONSTRUCTED_CONTENT)
        print("--- end reconstructed assistant stream ---")

        print("LOCAL_SSE_DONE=", LOCAL_SSE_DONE)
        print("LOCAL_SSE_FINISH_REASON=", LOCAL_SSE_FINISH_REASON)
        print("LOCAL_SSE_CONTENT_CHARS=", LOCAL_SSE_CONTENT_CHARS)
        print("LOCAL_SSE_REASONING_CHARS=", LOCAL_SSE_REASONING_CHARS)
        print("LOCAL_SSE_RAW_LINE_COUNT=", len(sse_raw_lines))
        print("LOCAL_SSE_PARSED_EVENT_COUNT=", len(sse_parsed_events))
        print("LOCAL_SSE_DRAFT_N=", LOCAL_SSE_DRAFT_N)
        print("LOCAL_SSE_DRAFT_N_ACCEPTED=", LOCAL_SSE_DRAFT_N_ACCEPTED)
        print("LOCAL_SSE_PREDICTED_PER_SECOND=", LOCAL_SSE_PREDICTED_PER_SECOND)

        if not LOCAL_SSE_DONE:
            LOCAL_SSE_SEMANTIC_FAILURE = "DONE_SENTINEL_NOT_SEEN"
        elif LOCAL_SSE_CONTENT_CHARS <= 0:
            LOCAL_SSE_SEMANTIC_FAILURE = "EMPTY_VISIBLE_CONTENT"
        elif LOCAL_SSE_FINISH_REASON not in {"stop", "length"}:
            LOCAL_SSE_SEMANTIC_FAILURE = "INVALID_FINISH_REASON"

        if LOCAL_SSE_SEMANTIC_FAILURE:
            LOCAL_SSE_FAILURE_CAPTURED = True
            print("LOCAL_SSE_SEMANTIC_FAILURE=" + LOCAL_SSE_SEMANTIC_FAILURE)
            capture_backend_forensics(
                "local-sse-stream-contract-failure",
                env=env,
            )
            print("LOCAL_SSE_DEMO=FAIL_CAPTURED")
        else:
            print("LOCAL_SSE_DEMO=PASS")

elif RUN_SSE_DEMO:
    INTERACTION_EVIDENCE["sse"] = {
        "status": "SKIPPED_DUE_TO_NONSTREAM_FAILURE"
    }
    write_interaction_evidence()
    print("LOCAL_SSE_DEMO=SKIPPED_DUE_TO_NONSTREAM_FAILURE")
else:
    INTERACTION_EVIDENCE["sse"] = {"status": "SKIPPED"}
    write_interaction_evidence()
    print("LOCAL_SSE_DEMO=SKIPPED")

SESSION["local_sse_http"] = LOCAL_SSE_STATUS
SESSION["local_sse_done"] = LOCAL_SSE_DONE
SESSION["local_sse_finish_reason"] = LOCAL_SSE_FINISH_REASON
SESSION["local_sse_content_chars"] = LOCAL_SSE_CONTENT_CHARS
SESSION["local_sse_reasoning_chars"] = LOCAL_SSE_REASONING_CHARS
SESSION["local_sse_failure_captured"] = LOCAL_SSE_FAILURE_CAPTURED
SESSION["local_sse_semantic_failure"] = LOCAL_SSE_SEMANTIC_FAILURE
SESSION["local_sse_draft_n"] = LOCAL_SSE_DRAFT_N
SESSION["local_sse_draft_n_accepted"] = LOCAL_SSE_DRAFT_N_ACCEPTED
SESSION["local_sse_predicted_per_second"] = LOCAL_SSE_PREDICTED_PER_SECOND

SESSION["local_sse_reasoning_strength"] = DEMO_REASONING_STRENGTH
SESSION["local_sse_visible_answer"] = (
    LOCAL_SSE_STATUS == 200
    and not LOCAL_SSE_FAILURE_CAPTURED
    and LOCAL_SSE_CONTENT_CHARS > 0
)


## 13. Run a deterministic semantic sanity task / Chạy bài semantic sanity xác định

**English.** The two infrastructure prompts above intentionally describe the deployment facts they ask the model to restate. This third request therefore checks an independent, deterministic task. It is **not** a quality benchmark; it simply proves that the served model can process a small calculation correctly through the same authenticated production path.

**Tiếng Việt.** Hai prompt hạ tầng phía trên cố ý nêu sẵn các deployment fact mà chúng yêu cầu model diễn đạt lại. Vì vậy request thứ ba kiểm tra một tác vụ độc lập, có đáp án xác định. Đây **không** phải benchmark chất lượng; nó chỉ chứng minh model đang serve có thể xử lý đúng một phép tính nhỏ qua cùng production path có xác thực.


In [ ]:
SEMANTIC_SANITY_HTTP = None
SEMANTIC_SANITY_RESULT = None
SEMANTIC_SANITY_EXPECTED = "104"
SEMANTIC_SANITY_GATE = "SKIPPED"
SEMANTIC_SANITY_DRAFT_N = None
SEMANTIC_SANITY_DRAFT_N_ACCEPTED = None
SEMANTIC_SANITY_PREDICTED_PER_SECOND = None

if RUN_SEMANTIC_SANITY:
    semantic_payload = {
        "model": "muse-glimmer-30B",
        "messages": [{
            "role": "user",
            "content": (
                "A production batch contains 128 requests. Five eighths of them "
                "succeed on the first attempt. Of the remaining requests, 24 "
                "succeed on retry. How many requests succeed in total? Return only "
                "the final integer and no other visible text."
            ),
        }],
        "max_tokens": DEMO_REQUEST_MAX_TOKENS,
        "chat_template_kwargs": {"reasoning_strength": DEMO_REASONING_STRENGTH},
        "stream": False,
    }
    semantic_url = f"{LOCAL_URL}/v1/chat/completions"
    semantic_headers = {**AUTH, "Content-Type": "application/json"}
    semantic_request_evidence = {
        "method": "POST",
        "url": semantic_url,
        "headers": redact_http_headers(semantic_headers),
        "json": semantic_payload,
    }

    print("=== SEMANTIC SANITY REQUEST (SANITIZED) ===")
    print(json.dumps(semantic_request_evidence, indent=2, ensure_ascii=False))

    semantic_started = time.monotonic()
    semantic_response = requests.post(
        semantic_url,
        headers=semantic_headers,
        json=semantic_payload,
        timeout=3600,
    )
    semantic_elapsed = time.monotonic() - semantic_started
    SEMANTIC_SANITY_HTTP = semantic_response.status_code

    semantic_response_headers = {
        str(k): str(v) for k, v in semantic_response.headers.items()
    }
    try:
        semantic_json = semantic_response.json()
    except Exception:
        semantic_json = None

    INTERACTION_EVIDENCE["semantic_sanity"] = {
        "request": semantic_request_evidence,
        "response": {
            "status_code": SEMANTIC_SANITY_HTTP,
            "headers": semantic_response_headers,
            "raw_body": semantic_response.text,
            "json": semantic_json,
        },
        "expected_visible_content": SEMANTIC_SANITY_EXPECTED,
    }
    write_interaction_evidence()

    print(f"SEMANTIC_SANITY_HTTP={SEMANTIC_SANITY_HTTP}")
    print(f"SEMANTIC_SANITY_ELAPSED_SECONDS={semantic_elapsed:.3f}")
    if SEMANTIC_SANITY_HTTP != 200:
        capture_backend_forensics(
            "semantic-sanity-http-failure", response=semantic_response, env=env
        )
        raise AssertionError(semantic_response.text)
    if not isinstance(semantic_json, dict):
        capture_backend_forensics(
            "semantic-sanity-json-failure", response=semantic_response, env=env
        )
        raise AssertionError("Semantic sanity response must be JSON.")

    choices = semantic_json.get("choices") or []
    choice = choices[0] if choices else {}
    message = choice.get("message") or {}
    SEMANTIC_SANITY_RESULT = str(message.get("content") or "").strip()
    semantic_timings = semantic_json.get("timings") or {}
    SEMANTIC_SANITY_DRAFT_N = semantic_timings.get("draft_n")
    SEMANTIC_SANITY_DRAFT_N_ACCEPTED = semantic_timings.get("draft_n_accepted")
    SEMANTIC_SANITY_PREDICTED_PER_SECOND = semantic_timings.get("predicted_per_second")

    print(f"SEMANTIC_SANITY_VISIBLE_RESULT={SEMANTIC_SANITY_RESULT!r}")
    print(f"SEMANTIC_SANITY_EXPECTED={SEMANTIC_SANITY_EXPECTED}")
    print(f"SEMANTIC_SANITY_DRAFT_N={SEMANTIC_SANITY_DRAFT_N}")
    print(f"SEMANTIC_SANITY_DRAFT_N_ACCEPTED={SEMANTIC_SANITY_DRAFT_N_ACCEPTED}")

    if choice.get("finish_reason") != "stop" or SEMANTIC_SANITY_RESULT != SEMANTIC_SANITY_EXPECTED:
        capture_backend_forensics(
            "semantic-sanity-contract-failure", response=semantic_response, env=env
        )
        raise AssertionError(
            "Semantic sanity contract failed: "
            f"finish_reason={choice.get('finish_reason')!r}, "
            f"observed={SEMANTIC_SANITY_RESULT!r}, expected={SEMANTIC_SANITY_EXPECTED!r}"
        )
    SEMANTIC_SANITY_GATE = "PASS"
else:
    INTERACTION_EVIDENCE["semantic_sanity"] = {"status": "SKIPPED"}
    write_interaction_evidence()

SESSION["semantic_sanity_http"] = SEMANTIC_SANITY_HTTP
SESSION["semantic_sanity_result"] = SEMANTIC_SANITY_RESULT
SESSION["semantic_sanity_expected"] = SEMANTIC_SANITY_EXPECTED
SESSION["semantic_sanity_gate"] = SEMANTIC_SANITY_GATE
SESSION["semantic_sanity_draft_n"] = SEMANTIC_SANITY_DRAFT_N
SESSION["semantic_sanity_draft_n_accepted"] = SEMANTIC_SANITY_DRAFT_N_ACCEPTED
SESSION["semantic_sanity_predicted_per_second"] = SEMANTIC_SANITY_PREDICTED_PER_SECOND
print(f"SEMANTIC_SANITY_GATE={SEMANTIC_SANITY_GATE}")


## 14. Summarize observed DFlash2 activity / Tổng hợp hoạt động DFlash2 quan sát được

**English.** A positive `draft_n` plus positive `draft_n_accepted` is runtime evidence that speculative decoding was active. Acceptance ratios and observed generation rates are reported as **single-run measurements only**. This notebook deliberately does not claim DFlash2 speedup because it does not run a matched DFlash2-OFF baseline.

**Tiếng Việt.** `draft_n > 0` cùng `draft_n_accepted > 0` là evidence runtime cho thấy speculative decoding thực sự hoạt động. Acceptance ratio và tốc độ generation quan sát được chỉ là **measurement của một run**. Notebook chủ động không tuyên bố speedup DFlash2 vì không chạy baseline đối chứng DFlash2-OFF tương ứng.


In [ ]:
def _positive_int_or_zero(value):
    try:
        return max(0, int(value))
    except (TypeError, ValueError):
        return 0


dflash_rows = []
if RUN_NONSTREAM_DEMO:
    dflash_rows.append((
        "nonstream",
        _positive_int_or_zero(LOCAL_NONSTREAM_DRAFT_N),
        _positive_int_or_zero(LOCAL_NONSTREAM_DRAFT_N_ACCEPTED),
        LOCAL_NONSTREAM_PREDICTED_PER_SECOND,
    ))
if RUN_SSE_DEMO:
    dflash_rows.append((
        "sse",
        _positive_int_or_zero(LOCAL_SSE_DRAFT_N),
        _positive_int_or_zero(LOCAL_SSE_DRAFT_N_ACCEPTED),
        LOCAL_SSE_PREDICTED_PER_SECOND,
    ))
if RUN_SEMANTIC_SANITY:
    dflash_rows.append((
        "semantic_sanity",
        _positive_int_or_zero(SEMANTIC_SANITY_DRAFT_N),
        _positive_int_or_zero(SEMANTIC_SANITY_DRAFT_N_ACCEPTED),
        SEMANTIC_SANITY_PREDICTED_PER_SECOND,
    ))

assert dflash_rows, "No generation path was enabled for DFlash2 activity evidence."

total_draft = 0
total_accepted = 0
dflash_summary = {}
for name, draft_n, accepted, predicted_per_second in dflash_rows:
    if draft_n <= 0 or accepted > draft_n:
        capture_backend_forensics("dflash2-activity-failure", env=env)
        raise AssertionError(
            f"{name}: invalid speculative counters draft_n={draft_n}, accepted={accepted}"
        )
    ratio = accepted / draft_n
    total_draft += draft_n
    total_accepted += accepted
    dflash_summary[name] = {
        "draft_n": draft_n,
        "draft_n_accepted": accepted,
        "acceptance_ratio": ratio,
        "predicted_per_second": predicted_per_second,
    }
    print(f"DFLASH2_{name.upper()}_DRAFT_N={draft_n}")
    print(f"DFLASH2_{name.upper()}_ACCEPTED={accepted}")
    print(f"DFLASH2_{name.upper()}_ACCEPTANCE_PERCENT={ratio * 100:.2f}")
    print(f"DFLASH2_{name.upper()}_OBSERVED_PREDICTED_PER_SECOND={predicted_per_second}")

if total_accepted <= 0:
    capture_backend_forensics("dflash2-zero-accepted-failure", env=env)
    raise AssertionError("No speculative draft token was accepted in this run.")
total_ratio = total_accepted / total_draft
print(f"DFLASH2_TOTAL_DRAFT_N={total_draft}")
print(f"DFLASH2_TOTAL_ACCEPTED={total_accepted}")
print(f"DFLASH2_TOTAL_ACCEPTANCE_PERCENT={total_ratio * 100:.2f}")
print("DFLASH2_SPEEDUP_CLAIM=NOT_MADE_NO_MATCHED_BASELINE")

SESSION["dflash2_observed"] = dflash_summary
SESSION["dflash2_total_draft_n"] = total_draft
SESSION["dflash2_total_accepted"] = total_accepted
SESSION["dflash2_total_acceptance_ratio"] = total_ratio
SESSION["dflash2_speedup_claim"] = "NOT_MADE_NO_MATCHED_BASELINE"
SESSION["dflash2_activity_gate"] = "PASS"
print("DFLASH2_ACTIVITY_GATE=PASS")


## Public Quick Tunnel transport qualification / Qualification transport qua Public Quick Tunnel

### English

The core production demo has already exercised the real model, DFlash2, authenticated gateway, non-streaming generation, and SSE through loopback.

This step qualifies only the **external transport**. Quick Tunnel DNS can be transient, so the notebook retries DNS + HTTP for up to `PUBLIC_DNS_RETRY_SECONDS` while re-checking:

- external exposure remains `ready`;
- tunnel remains owned/alive;
- the public URL has not changed;
- public unauthenticated `/v1/models` returns `401`;
- public authenticated `/ready` returns `200`.

A persistent public DNS/transport failure is recorded as a **transport blocker**, not a model/runtime blocker. The notebook continues to evidence and cleanup instead of aborting.

---

### Tiếng Việt

Core production demo đã kiểm tra model thực tế, DFlash2, authenticated gateway, non-streaming generation và SSE thông qua loopback.

Bước này chỉ qualification **external transport**. DNS của Quick Tunnel có thể gặp lỗi tạm thời, vì vậy notebook sẽ retry DNS + HTTP trong tối đa `PUBLIC_DNS_RETRY_SECONDS` đồng thời kiểm tra lại:

- external exposure vẫn ở trạng thái `ready`;
- tunnel vẫn thuộc quyền quản lý của demo và còn hoạt động;
- public URL không thay đổi;
- request công khai tới `/v1/models` khi không xác thực trả về `401`;
- request công khai có xác thực tới `/ready` trả về `200`.

Nếu lỗi public DNS/transport kéo dài, notebook ghi nhận đó là **transport blocker**, không phải model/runtime blocker. Notebook vẫn tiếp tục ghi evidence và cleanup thay vì dừng ngay.


## 15. Qualify the optional public Quick Tunnel transport / Qualification transport Quick Tunnel công khai tùy chọn

**English:** Only after the local core succeeds, this cell stops the local proxy exposure and starts `EXPOSURE_MODE=quick` as a separate transport phase. It retries public DNS/HTTP checks and classifies persistent tunnel failures independently from model/runtime health. A core PASS with public failure remains a transport-only blocker; the model core is not invalidated.

**Tiếng Việt:** Chỉ sau khi local core thành công, cell này dừng local proxy exposure và khởi động `EXPOSURE_MODE=quick` như một phase transport tách biệt. Cell retry các kiểm tra DNS/HTTP công khai và phân loại lỗi tunnel kéo dài độc lập với sức khỏe model/runtime. Core PASS nhưng public FAIL vẫn chỉ là transport blocker; kết quả core model không bị vô hiệu hóa.


In [ ]:
PUBLIC_QUICK_TUNNEL = "NOT_RUN"
PUBLIC_FAILURE_CLASS = None
PUBLIC_LAST_ERROR = None
PUBLIC_DNS_ADDRESSES = []
PUBLIC_UNAUTH_MODELS_HTTP = None
PUBLIC_AUTH_READY_HTTP = None
PUBLIC_ATTEMPTS = 0
PUBLIC_QUICK_START_RC = None
PUBLIC_QUICK_START_OUTPUT_TAIL = None
PUBLIC_URL = None
PUBLIC_STOP_PROXY_RC = None

core_generation_ok = (
    (not RUN_NONSTREAM_DEMO or (
        LOCAL_NONSTREAM_STATUS == 200
        and not LOCAL_NONSTREAM_FAILURE_CAPTURED
    ))
    and
    (not RUN_SSE_DEMO or (
        LOCAL_SSE_STATUS == 200
        and LOCAL_SSE_DONE
        and not LOCAL_SSE_FAILURE_CAPTURED
    ))
)

if not core_generation_ok:
    PUBLIC_QUICK_TUNNEL = "NOT_RUN"
    PUBLIC_FAILURE_CLASS = "CORE_GENERATION_FAILURE"
    print("PUBLIC_QUICK_TUNNEL=NOT_RUN")
    print("PUBLIC_FAILURE_CLASS=CORE_GENERATION_FAILURE")
    print("PUBLIC_TRANSPORT_QUALIFICATION=SKIPPED_DUE_TO_CORE_FAILURE")

else:
    # Transition cleanly from proxy-only core mode to quick public mode.
    proxy_env, _ = build_canonical_demo_env(
        token,
        cloudflared,
        exposure_mode="proxy",
    )
    stop_proxy = run(
        ["./external.sh", "stop"],
        cwd=WORKDIR,
        env=proxy_env,
        check=False,
    )
    PUBLIC_STOP_PROXY_RC = stop_proxy.returncode
    print(f"PUBLIC_PHASE_PROXY_STOP_RC={PUBLIC_STOP_PROXY_RC}")

    quick_env, _ = build_canonical_demo_env(
        token,
        cloudflared,
        exposure_mode="quick",
    )

    quick_preflight = run(
        ["./external.sh", "preflight"],
        cwd=WORKDIR,
        env=quick_env,
        check=False,
    )
    print(f"PUBLIC_QUICK_PREFLIGHT_RC={quick_preflight.returncode}")

    quick_start = run(
        ["./external.sh", "start"],
        cwd=WORKDIR,
        env=quick_env,
        check=False,
    )
    PUBLIC_QUICK_START_RC = quick_start.returncode
    quick_output = quick_start.stdout or ""
    PUBLIC_QUICK_START_OUTPUT_TAIL = "\n".join(
        quick_output.splitlines()[-60:]
    )

    print(f"PUBLIC_QUICK_START_RC={PUBLIC_QUICK_START_RC}")

    if PUBLIC_QUICK_START_RC != 0:
        PUBLIC_QUICK_TUNNEL = "FAIL"

        if (
            "timed out waiting for authenticated public readiness" in quick_output
            and "LAST_UNAUTH_CODE=000" in quick_output
            and "LAST_AUTH_CODE=000" in quick_output
        ):
            PUBLIC_FAILURE_CLASS = "PUBLIC_READINESS_UNREACHABLE"
        elif "timed out waiting for authenticated public readiness" in quick_output:
            PUBLIC_FAILURE_CLASS = "PUBLIC_READINESS_TIMEOUT"
        elif "Quick Tunnel" in quick_output:
            PUBLIC_FAILURE_CLASS = "QUICK_TUNNEL_START_FAILURE"
        else:
            PUBLIC_FAILURE_CLASS = "PUBLIC_START_FAILURE"

        PUBLIC_LAST_ERROR = PUBLIC_QUICK_START_OUTPUT_TAIL

        print(f"PUBLIC_FAILURE_CLASS={PUBLIC_FAILURE_CLASS}")
        print("PUBLIC_TRANSPORT_BLOCKER=YES")
        print("MODEL_RUNTIME_BLOCKER=NO")
        if PUBLIC_LAST_ERROR:
            print("PUBLIC_QUICK_START_OUTPUT_TAIL_BEGIN")
            print(PUBLIC_LAST_ERROR)
            print("PUBLIC_QUICK_START_OUTPUT_TAIL_END")

        # Best-effort fail-closed exposure cleanup; backend remains owned/running.
        run(
            ["./external.sh", "stop"],
            cwd=WORKDIR,
            env=quick_env,
            check=False,
        )

    else:
        # Verify the newly launched gateway still uses effective Context/output values.
        verify_auth_proxy_runtime_limits(
            SESSION["effective_context_size"], SESSION["effective_max_output_tokens"]
        )

        PUBLIC_URL = run(
            ["./external.sh", "endpoint"],
            cwd=WORKDIR,
            env=quick_env,
        ).stdout.strip().rstrip("/")

        parsed_public = urllib.parse.urlsplit(PUBLIC_URL)
        assert parsed_public.scheme == "https" and parsed_public.hostname
        public_host = parsed_public.hostname

        deadline = time.monotonic() + float(PUBLIC_DNS_RETRY_SECONDS)

        while True:
            PUBLIC_ATTEMPTS += 1
            try:
                infos = socket.getaddrinfo(
                    public_host,
                    443,
                    type=socket.SOCK_STREAM,
                )
                PUBLIC_DNS_ADDRESSES = sorted({
                    item[4][0]
                    for item in infos
                    if item and len(item) >= 5 and item[4]
                })

                unauth = requests.get(
                    f"{PUBLIC_URL}/v1/models",
                    timeout=PUBLIC_HTTP_TIMEOUT_SECONDS,
                    headers={"Connection": "close"},
                )
                PUBLIC_UNAUTH_MODELS_HTTP = unauth.status_code

                auth_ready = requests.get(
                    f"{PUBLIC_URL}/ready",
                    headers={**AUTH, "Connection": "close"},
                    timeout=PUBLIC_HTTP_TIMEOUT_SECONDS,
                )
                PUBLIC_AUTH_READY_HTTP = auth_ready.status_code

                if (
                    PUBLIC_UNAUTH_MODELS_HTTP == 401
                    and PUBLIC_AUTH_READY_HTTP == 200
                ):
                    PUBLIC_QUICK_TUNNEL = "PASS"
                    PUBLIC_FAILURE_CLASS = None
                    PUBLIC_LAST_ERROR = None
                    break

                PUBLIC_FAILURE_CLASS = "PUBLIC_HTTP_CONTRACT"
                PUBLIC_LAST_ERROR = (
                    f"unauth_models={PUBLIC_UNAUTH_MODELS_HTTP} "
                    f"auth_ready={PUBLIC_AUTH_READY_HTTP}"
                )

            except socket.gaierror as exc:
                PUBLIC_FAILURE_CLASS = "DNS_RESOLUTION"
                PUBLIC_LAST_ERROR = f"{type(exc).__name__}: {exc}"
            except requests.exceptions.Timeout as exc:
                PUBLIC_FAILURE_CLASS = "HTTP_TIMEOUT"
                PUBLIC_LAST_ERROR = f"{type(exc).__name__}: {exc}"
            except requests.exceptions.ConnectionError as exc:
                text = str(exc)
                if (
                    "NameResolutionError" in text
                    or "Failed to resolve" in text
                    or "Name or service not known" in text
                ):
                    PUBLIC_FAILURE_CLASS = "DNS_RESOLUTION"
                else:
                    PUBLIC_FAILURE_CLASS = "CONNECTION_ERROR"
                PUBLIC_LAST_ERROR = f"{type(exc).__name__}: {exc}"
            except requests.RequestException as exc:
                PUBLIC_FAILURE_CLASS = "HTTP_REQUEST_ERROR"
                PUBLIC_LAST_ERROR = f"{type(exc).__name__}: {exc}"

            if time.monotonic() >= deadline:
                PUBLIC_QUICK_TUNNEL = "FAIL"
                break

            remaining = max(0.0, deadline - time.monotonic())
            sleep_for = min(float(PUBLIC_RETRY_INTERVAL_SECONDS), remaining)
            if sleep_for > 0:
                print(
                    f"PUBLIC_TRANSPORT_RETRY attempt={PUBLIC_ATTEMPTS} "
                    f"class={PUBLIC_FAILURE_CLASS} sleep={sleep_for:.1f}s"
                )
                time.sleep(sleep_for)

        print(f"PUBLIC_URL={PUBLIC_URL}")
        print(f"PUBLIC_QUICK_TUNNEL={PUBLIC_QUICK_TUNNEL}")
        print(f"PUBLIC_ATTEMPTS={PUBLIC_ATTEMPTS}")
        print(f"PUBLIC_DNS_ADDRESSES={PUBLIC_DNS_ADDRESSES}")
        print(f"PUBLIC_UNAUTH_V1_MODELS_HTTP={PUBLIC_UNAUTH_MODELS_HTTP}")
        print(f"PUBLIC_AUTH_READY_HTTP={PUBLIC_AUTH_READY_HTTP}")
        print(f"PUBLIC_FAILURE_CLASS={PUBLIC_FAILURE_CLASS}")

        if PUBLIC_LAST_ERROR:
            print(f"PUBLIC_LAST_ERROR={PUBLIC_LAST_ERROR[:1500]}")

        if PUBLIC_QUICK_TUNNEL == "PASS":
            print("PUBLIC_TRANSPORT_BLOCKER=NO")
        else:
            print("PUBLIC_TRANSPORT_BLOCKER=YES")
            print("MODEL_RUNTIME_BLOCKER=NO")

SESSION["public_quick_tunnel"] = PUBLIC_QUICK_TUNNEL
SESSION["public_failure_class"] = PUBLIC_FAILURE_CLASS
SESSION["public_quick_start_rc"] = PUBLIC_QUICK_START_RC
SESSION["public_proxy_stop_rc"] = PUBLIC_STOP_PROXY_RC
SESSION["public_attempts"] = PUBLIC_ATTEMPTS
SESSION["public_dns_address_count"] = len(PUBLIC_DNS_ADDRESSES)
SESSION["public_unauth_v1_models_http"] = PUBLIC_UNAUTH_MODELS_HTTP
SESSION["public_auth_ready_http"] = PUBLIC_AUTH_READY_HTTP
SESSION["public_endpoint_host"] = (
    urllib.parse.urlsplit(PUBLIC_URL).hostname
    if PUBLIC_URL
    else None
)


## Concurrency behavior / Hành vi concurrency

### English

`v1.0.0` intentionally has one active inference slot (`PARALLEL_SLOTS=1`). Overlapping inference is expected to return HTTP `429` with `muse_demo_busy`.

This production-demo notebook does **not** automatically generate overlapping real-model requests because doing so wastes GPU quota. The deterministic contract suite and publication evidence already cover admission behavior. The notebook focuses real GPU work on one local non-streaming generation and one local SSE generation.

---

### Tiếng Việt

`v1.0.0` chủ động chỉ sử dụng một inference slot hoạt động (`PARALLEL_SLOTS=1`). Khi các inference request chồng lấn nhau, hành vi mong đợi là trả về HTTP `429` với mã `muse_demo_busy`.

Production-demo notebook này **không** tự động tạo các real-model request chồng lấn vì việc đó làm lãng phí GPU quota. Deterministic contract suite và publication evidence đã bao phủ hành vi admission này. Notebook tập trung phần GPU thực tế vào một local non-streaming generation và một local SSE generation.


## 16. Write secret-free verdict evidence and bind the full interaction transcript / Ghi verdict evidence không chứa secret và liên kết full interaction transcript

**English:** This cell consolidates all prior gates into `/kaggle/working/muse-glimmer-30b-v1.0.0-demo-evidence.json` and validates the reviewer-facing interaction transcript at `/kaggle/working/muse-glimmer-30b-v1.0.0-interaction-evidence.json`. The interaction artifact contains the complete sanitized chat-completions request/response evidence for both non-stream and SSE; the Bearer credential is never written. Its SHA-256 is bound into the compact verdict evidence. Core PASS now requires not only fork commit identity, canonical environment, local auth and visible model output, but also `INTERACTION_EVIDENCE_GATE=PASS`.

**Tiếng Việt:** Cell này tổng hợp tất cả gate trước đó vào `/kaggle/working/muse-glimmer-30b-v1.0.0-demo-evidence.json` và xác minh reviewer-facing interaction transcript tại `/kaggle/working/muse-glimmer-30b-v1.0.0-interaction-evidence.json`. Interaction artifact chứa đầy đủ evidence request/response đã sanitize cho cả non-stream và SSE; Bearer credential tuyệt đối không được ghi ra. SHA-256 của artifact được liên kết vào verdict evidence gọn. Core PASS giờ yêu cầu không chỉ fork commit identity, canonical environment, local auth và visible model output mà còn phải có `INTERACTION_EVIDENCE_GATE=PASS`.


In [ ]:
SESSION["completed_at_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat()

SESSION["publication_identity_verified"] = (
    SESSION.get("source_ref") == SOURCE_REF
    and SESSION.get("resolved_source_sha") == SESSION.get("actual_head")
    and SESSION.get("release_version") == EXPECTED_VERSION
    and SESSION.get("worktree_clean") is True
    and SESSION.get("source_manifest") == "PASS"
)


interaction_evidence_gate = "FAIL"
interaction_evidence_sha256 = None
interaction_evidence_size_bytes = None
interaction_nonstream_complete = False
interaction_sse_complete = False
interaction_semantic_complete = False
interaction_secret_scan = "FAIL"

if INTERACTION_EVIDENCE_PATH.exists():
    interaction_bytes = INTERACTION_EVIDENCE_PATH.read_bytes()
    interaction_text = interaction_bytes.decode("utf-8")
    interaction_doc = json.loads(interaction_text)

    interaction_nonstream = interaction_doc.get("nonstream") or {}
    interaction_sse = interaction_doc.get("sse") or {}
    interaction_semantic = interaction_doc.get("semantic_sanity") or {}

    interaction_nonstream_complete = all([
        isinstance(interaction_nonstream.get("request"), dict),
        isinstance(interaction_nonstream.get("response"), dict),
        interaction_nonstream.get("response", {}).get("status_code") == 200,
        isinstance(interaction_nonstream.get("response", {}).get("raw_body"), str),
        isinstance(interaction_nonstream.get("response", {}).get("json"), dict),
    ])

    sse_response = interaction_sse.get("response") or {}
    interaction_sse_complete = all([
        isinstance(interaction_sse.get("request"), dict),
        isinstance(sse_response, dict),
        sse_response.get("status_code") == 200,
        isinstance(sse_response.get("raw_sse_lines"), list),
        len(sse_response.get("raw_sse_lines") or []) > 0,
        isinstance(sse_response.get("parsed_events"), list),
        len(sse_response.get("parsed_events") or []) > 0,
        sse_response.get("done_seen") is True,
        isinstance(sse_response.get("reconstructed"), dict),
        bool((sse_response.get("reconstructed") or {}).get("content")),
    ])

    if RUN_SEMANTIC_SANITY:
        semantic_response = interaction_semantic.get("response") or {}
        semantic_json = semantic_response.get("json") or {}
        semantic_choices = semantic_json.get("choices") or []
        semantic_choice = semantic_choices[0] if semantic_choices else {}
        semantic_message = semantic_choice.get("message") or {}
        interaction_semantic_complete = all([
            isinstance(interaction_semantic.get("request"), dict),
            semantic_response.get("status_code") == 200,
            str(semantic_message.get("content") or "").strip() == "104",
            semantic_choice.get("finish_reason") == "stop",
        ])
    else:
        interaction_semantic_complete = interaction_semantic.get("status") == "SKIPPED"

    assert token not in interaction_text, (
        "Bearer credential leaked into interaction evidence artifact."
    )
    interaction_secret_scan = "PASS"

    interaction_evidence_sha256 = hashlib.sha256(interaction_bytes).hexdigest()
    interaction_evidence_size_bytes = len(interaction_bytes)

    if all([
        interaction_nonstream_complete,
        interaction_sse_complete,
        interaction_semantic_complete,
        interaction_secret_scan == "PASS",
    ]):
        interaction_evidence_gate = "PASS"

SESSION["interaction_evidence_path"] = str(INTERACTION_EVIDENCE_PATH)
SESSION["interaction_evidence_sha256"] = interaction_evidence_sha256
SESSION["interaction_evidence_size_bytes"] = interaction_evidence_size_bytes
SESSION["interaction_nonstream_complete"] = interaction_nonstream_complete
SESSION["interaction_sse_complete"] = interaction_sse_complete
SESSION["interaction_semantic_complete"] = interaction_semantic_complete
SESSION["interaction_secret_scan"] = interaction_secret_scan
SESSION["interaction_evidence_gate"] = interaction_evidence_gate

print(f"INTERACTION_EVIDENCE_PATH={INTERACTION_EVIDENCE_PATH}")
print(f"INTERACTION_EVIDENCE_SHA256={interaction_evidence_sha256}")
print(f"INTERACTION_EVIDENCE_SIZE_BYTES={interaction_evidence_size_bytes}")
print(f"INTERACTION_NONSTREAM_COMPLETE={interaction_nonstream_complete}")
print(f"INTERACTION_SSE_COMPLETE={interaction_sse_complete}")
print(f"INTERACTION_SEMANTIC_COMPLETE={interaction_semantic_complete}")
print(f"INTERACTION_SECRET_SCAN={interaction_secret_scan}")
print(f"INTERACTION_EVIDENCE_GATE={interaction_evidence_gate}")


core_pass = all([
    SESSION.get("publication_identity_verified"),
    SESSION.get("cloudflared_sha256_verified") is True,
    SESSION.get("gpu_topology_gate") == "PASS",
    SESSION.get("model_sha256_gate") == "PASS",
    SESSION.get("dual_gpu_residency_gate") == "PASS",
    SESSION.get("backend_provenance") == "PASS",
    SESSION.get("dflash2_activity_gate") == "PASS",
    (not RUN_SEMANTIC_SANITY) or SESSION.get("semantic_sanity_gate") == "PASS",
    SESSION.get("interaction_evidence_gate") == "PASS",
    SESSION.get("canonical_environment") == "PASS",
    SESSION.get("demo_request_budget_gate") == "PASS",
    SESSION.get("demo_reasoning_strength") == "high",
    SESSION.get("gateway_idle_timeout_seconds") == 180,
    SESSION.get("core_exposure_mode") == "proxy",
    SESSION.get("gateway_runtime_limits_process_gate") == "PASS",
    SESSION.get("post_start_source_manifest") == "PASS",
    SESSION.get("rerun_recovery") == "PASS",
    SESSION.get("port_8088_free_before_start") is True,
    SESSION.get("port_8090_free_before_start") is True,
    SESSION.get("local_unauth_v1_models_http") == 401,
    SESSION.get("local_auth_ready_http") == 200,
    SESSION.get("local_auth_v1_models_http") == 200,
    (not RUN_NONSTREAM_DEMO) or (
        SESSION.get("local_nonstream_http") == 200
        and SESSION.get("local_nonstream_failure_captured") is False
        and (SESSION.get("local_nonstream_content_chars") or 0) > 0
        and SESSION.get("local_nonstream_visible_answer") is True
    ),
    (not RUN_SSE_DEMO) or (
        SESSION.get("local_sse_done") is True
        and SESSION.get("local_sse_failure_captured") is False
        and (SESSION.get("local_sse_content_chars") or 0) > 0
        and SESSION.get("local_sse_visible_answer") is True
    ),
])

public_pass = SESSION.get("public_quick_tunnel") == "PASS"

SESSION["visible_answer_gate"] = "PASS" if all([
    (not RUN_NONSTREAM_DEMO) or SESSION.get("local_nonstream_visible_answer") is True,
    (not RUN_SSE_DEMO) or SESSION.get("local_sse_visible_answer") is True,
]) else "FAIL"

SESSION["core_production_demo"] = "PASS" if core_pass else "FAIL"
SESSION["public_quick_tunnel_verdict"] = "PASS" if public_pass else "FAIL"
SESSION["model_runtime_blocker"] = "NO" if core_pass else "YES"
SESSION["transport_blocker"] = "NO" if public_pass else "YES"

if core_pass and public_pass:
    SESSION["overall_production_demo"] = "PASS"
elif core_pass:
    SESSION["overall_production_demo"] = "PARTIAL_TRANSPORT_BLOCKER"
else:
    SESSION["overall_production_demo"] = "FAIL"

for forbidden_key in ("token", "MUSE_API_TOKEN", "prompt", "completion"):
    assert forbidden_key not in SESSION

EVIDENCE_PATH.write_text(
    json.dumps(SESSION, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print(EVIDENCE_PATH)
print(json.dumps(SESSION, indent=2, ensure_ascii=False))

print(f"VISIBLE_ANSWER_GATE={SESSION['visible_answer_gate']}")
print(f"CORE_PRODUCTION_DEMO={SESSION['core_production_demo']}")
print(f"PUBLIC_QUICK_TUNNEL={SESSION['public_quick_tunnel_verdict']}")
print(f"MODEL_RUNTIME_BLOCKER={SESSION['model_runtime_blocker']}")
print(f"TRANSPORT_BLOCKER={SESSION['transport_blocker']}")
print(f"OVERALL_PRODUCTION_DEMO={SESSION['overall_production_demo']}")

print("PRODUCTION_DEMO_EVIDENCE=PASS" if core_pass else "PRODUCTION_DEMO_EVIDENCE=FAIL_CAPTURED")


## Cleanup / Dọn dẹp

### English

Cleanup runs after evidence even when Quick Tunnel qualification ends with a transport blocker.

With the defaults:

- `AUTO_CLEANUP_AT_END=True` stops the tunnel/gateway first and then the owned backend.
- `AUTO_DELETE_GENERATED_TOKEN_AT_END=True` removes the notebook-generated token file.

A public DNS/transport failure therefore does not leave the demo intentionally running just because the public qualification did not pass.

---

### Tiếng Việt

Cleanup vẫn chạy sau khi evidence được ghi, kể cả khi qualification của Quick Tunnel kết thúc với một transport blocker.

Với cấu hình mặc định:

- `AUTO_CLEANUP_AT_END=True` dừng tunnel/gateway trước, sau đó dừng backend thuộc quyền quản lý của demo.
- `AUTO_DELETE_GENERATED_TOKEN_AT_END=True` xóa file token do notebook tự tạo.

Vì vậy, một lỗi public DNS/transport sẽ không khiến demo bị cố ý để tiếp tục chạy chỉ vì public qualification chưa đạt.


## Publication evidence contract / Contract publication evidence

**English.** The final cell produces a human-readable reviewer summary plus machine-checkable artifacts. The Quick Tunnel URL printed earlier is historical once cleanup succeeds; reviewers should not expect that temporary URL to remain live. Full request/response evidence is preserved in the redacted interaction JSON, while the notebook summary surfaces the gates that matter for review.

The DFlash2 counters are proof of observed speculative-decoding activity. They are **not** a DFlash2-vs-baseline benchmark. A performance speedup claim belongs in a separate controlled benchmark notebook.

**Tiếng Việt.** Cell cuối tạo summary dễ đọc cho reviewer cùng các artifact có thể kiểm chứng bằng máy. Quick Tunnel URL đã in phía trên trở thành URL lịch sử sau khi cleanup thành công; reviewer không nên kỳ vọng endpoint tạm thời đó còn hoạt động. Full request/response evidence được giữ trong interaction JSON đã redact, còn notebook summary surface các gate quan trọng nhất cho review.

Counter DFlash2 là bằng chứng cho speculative decoding đã thực sự hoạt động. Chúng **không** phải benchmark DFlash2-vs-baseline. Claim về speedup nên nằm trong một notebook benchmark có kiểm soát riêng.


## 17. Clean up owned processes and report only current-run forensics / Dọn dẹp process thuộc demo và chỉ báo forensic của run hiện tại

**English:** This final cell runs canonical `stop-all`, deletes any notebook-generated token file when configured, updates the evidence JSON with cleanup state, and asserts only real cleanup failure. A semantic FAIL no longer turns the cleanup cell red. The cell reports a forensic ZIP only when `FORENSIC_CAPTURE_COUNT` for this run is greater than zero; otherwise it prints `FINAL_FORENSIC_ZIP=NOT_CREATED_THIS_RUN`. It then prints `FINAL_NOTEBOOK_RESULT=PASS`, `PARTIAL_TRANSPORT_BLOCKER`, or `FAIL`, followed by `NOTEBOOK_EXECUTION_COMPLETED=PASS`.

**Tiếng Việt:** Cell cuối này chạy canonical `stop-all`, xóa token file do notebook tự tạo khi được cấu hình, cập nhật evidence JSON với trạng thái cleanup và chỉ assert khi cleanup thực sự thất bại. Semantic FAIL không còn làm cleanup cell đỏ. Cell chỉ báo forensic ZIP khi `FORENSIC_CAPTURE_COUNT` của chính run này lớn hơn 0; nếu không sẽ in `FINAL_FORENSIC_ZIP=NOT_CREATED_THIS_RUN`. Sau đó notebook in `FINAL_NOTEBOOK_RESULT=PASS`, `PARTIAL_TRANSPORT_BLOCKER` hoặc `FAIL`, rồi `NOTEBOOK_EXECUTION_COMPLETED=PASS`.


In [ ]:
generated_token_deleted = False
stop_rc = None
cleanup_gate = "SKIPPED"

if AUTO_CLEANUP_AT_END:
    print("Stopping public exposure + owned backend...")
    cleanup_summary = muse_stop(delete_generated_token=AUTO_DELETE_GENERATED_TOKEN_AT_END)
    stop_rc = cleanup_summary.get("canonical_stop_rc")
    generated_token_deleted = cleanup_summary.get("generated_token_deleted", False)

    if stop_rc not in (None, 0):
        raise RuntimeError(f"Canonical stop-all failed with rc={stop_rc}")

    cleanup_gate = "PASS"
    print("AUTO_CLEANUP=PASS")
    if SESSION.get("secret_source") == "auto_generated_file":
        assert not GENERATED_TOKEN_FILE.exists(), "Generated token file survived cleanup."
        print("GENERATED_TOKEN_CLEANUP=PASS")
    else:
        print("GENERATED_TOKEN_FILE=NOT_CREATED_BY_NOTEBOOK")
        print("GENERATED_TOKEN_CLEANUP=PASS")
else:
    cleanup_summary = {
        "canonical_stop_attempted": False,
        "canonical_stop_rc": None,
        "generated_token_deleted": False,
    }
    print("AUTO_CLEANUP=SKIPPED")
    print("The temporary public endpoint is still operator-controlled.")
    print("When finished, execute: muse_stop()")
    if SESSION.get("secret_source") == "auto_generated_file":
        print(f"TOKEN_FILE={GENERATED_TOKEN_FILE}")
        print("IMPORTANT: Delete the generated credential after stopping the demo.")

SESSION["cleanup_stop_rc"] = stop_rc
SESSION["cleanup_gate"] = cleanup_gate
SESSION["generated_token_deleted_at_cleanup"] = generated_token_deleted
SESSION["public_url_lifecycle"] = (
    "HISTORICAL_TEST_ENDPOINT_STOPPED"
    if AUTO_CLEANUP_AT_END
    else "TEMPORARILY_OPERATOR_CONTROLLED"
)

# The verdict JSON is finalized only after cleanup so its checksum binds the actual final state.
SESSION["finalized_at_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
EVIDENCE_PATH.write_text(
    json.dumps(SESSION, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)


def pct(value):
    if value is None:
        return "n/a"
    return f"{float(value) * 100:.2f}%"


def fmt_rate(value):
    if value is None:
        return "n/a"
    return f"{float(value):.2f} tok/s"


core_verdict = SESSION.get("core_production_demo", "UNKNOWN")
public_verdict = SESSION.get("public_quick_tunnel_verdict", "UNKNOWN")
overall_verdict = SESSION.get("overall_production_demo", "UNKNOWN")
if overall_verdict == "PASS":
    final_notebook_result = "PASS"
elif overall_verdict == "PARTIAL_TRANSPORT_BLOCKER":
    final_notebook_result = "PARTIAL_TRANSPORT_BLOCKER"
else:
    final_notebook_result = "FAIL"

# Cross-check aggregate DFlash2 counters before presenting them to reviewers.
observed_dflash = SESSION.get("dflash2_observed") or {}
derived_total_draft = sum(
    int((row or {}).get("draft_n", 0)) for row in observed_dflash.values()
)
derived_total_accepted = sum(
    int((row or {}).get("draft_n_accepted", 0)) for row in observed_dflash.values()
)
assert derived_total_draft == int(SESSION.get("dflash2_total_draft_n", 0))
assert derived_total_accepted == int(SESSION.get("dflash2_total_accepted", 0))
print("DFLASH2_SUMMARY_ARITHMETIC_GATE=PASS")

dflash_detail_lines = []
for key, label in [
    ("nonstream", "Non-stream"),
    ("sse", "SSE"),
    ("semantic_sanity", "Semantic sanity"),
]:
    row = observed_dflash.get(key)
    if not row:
        continue
    dflash_detail_lines.extend([
        f"  {label} draft/accepted:".ljust(34)
        + f"{row.get('draft_n')} / {row.get('draft_n_accepted')}",
        f"  {label} observed rate:".ljust(34)
        + fmt_rate(row.get("predicted_per_second")),
    ])

credential_lines = [
    f"  Credential source:           {SESSION.get('secret_source')}",
]
if SESSION.get("secret_source") == "auto_generated_file":
    credential_lines.extend([
        "  Temporary credential scope: ephemeral runtime file only",
        f"  Runtime credential mode:     {SESSION.get('secret_file_mode')}",
        "  Included in publication:     NO",
        f"  Deleted during cleanup:      {'PASS' if SESSION.get('generated_token_deleted_at_cleanup') else 'FAIL'}",
    ])
else:
    credential_lines.extend([
        "  Temporary credential scope: not applicable (operator-managed secret)",
        "  Included in publication:     NO",
    ])

summary_lines = [
    "Muse-Glimmer-30B v1.0.0 — Publication Reviewer Summary",
    "=" * 62,
    f"Run ID:                       {SESSION.get('run_id')}",
    f"Release version:              {SESSION.get('release_version')}",
    f"Source ref:                   {SESSION.get('source_ref')}",
    f"Resolved source SHA:           {SESSION.get('actual_head')}",
    f"Resolved source HEAD:         {SESSION.get('actual_head')}",
    f"Source manifest:              {SESSION.get('source_manifest')}",
    f"Cloudflared pinned version:    {SESSION.get('cloudflared_expected_version')}",
    f"Cloudflared SHA-256 verified:  {SESSION.get('cloudflared_sha256_verified')}",
    f"Cloudflared observed SHA-256:  {SESSION.get('cloudflared_sha256')}",
    "",
    "Hardware / runtime",
    f"  Kaggle topology:             NVIDIA Tesla T4 x2 ({SESSION.get('gpu_topology_gate')})",
    f"  Dual-GPU memory residency:   {SESSION.get('dual_gpu_residency_gate')}",
    f"  GPU0 VRAM delta:             {(SESSION.get('gpu_residency_deltas_mib') or {}).get('0')} MiB",
    f"  GPU1 VRAM delta:             {(SESSION.get('gpu_residency_deltas_mib') or {}).get('1')} MiB",
    f"  Backend provenance:          {SESSION.get('backend_provenance')}",
    f"  Model SHA-256 gate:          {SESSION.get('model_sha256_gate')}",
    f"  Target observed SHA-256:     {SESSION.get('target_observed_sha256')}",
    f"  DFlash2 observed SHA-256:    {SESSION.get('draft_observed_sha256')}",
    "",
    "Authenticated API",
    f"  Unauthenticated /v1/models: HTTP {SESSION.get('local_unauth_v1_models_http')}",
    f"  Authenticated /ready:        HTTP {SESSION.get('local_auth_ready_http')}",
    f"  Authenticated /v1/models:    HTTP {SESSION.get('local_auth_v1_models_http')}",
    f"  Non-stream generation:       HTTP {SESSION.get('local_nonstream_http')} / finish={SESSION.get('local_nonstream_finish_reason')}",
    f"  SSE generation:              HTTP {SESSION.get('local_sse_http')} / DONE={SESSION.get('local_sse_done')} / finish={SESSION.get('local_sse_finish_reason')}",
    f"  Semantic sanity:             {SESSION.get('semantic_sanity_gate')} (expected=104, observed={SESSION.get('semantic_sanity_result')})",
    "",
    "Observed DFlash2 activity (single-run measurements; no speedup claim)",
    *dflash_detail_lines,
    f"  Total across {len(observed_dflash)} requests:".ljust(34)
    + f"{SESSION.get('dflash2_total_draft_n')} / {SESSION.get('dflash2_total_accepted')}",
    f"  Total acceptance ratio:".ljust(34)
    + pct(SESSION.get("dflash2_total_acceptance_ratio")),
    f"  DFlash2 activity gate:".ljust(34)
    + str(SESSION.get("dflash2_activity_gate")),
    "",
    "Evidence / security",
    f"  Interaction evidence:        {SESSION.get('interaction_evidence_gate')}",
    f"  Interaction secret scan:     {SESSION.get('interaction_secret_scan')}",
    *credential_lines,
    f"  Cleanup gate:                {SESSION.get('cleanup_gate')}",
    f"  Public URL lifecycle:        {SESSION.get('public_url_lifecycle')}",
    "",
    "Final verdict",
    f"  Core production demo:        {core_verdict}",
    f"  Public Quick Tunnel:         {public_verdict}",
    f"  Overall production demo:     {overall_verdict}",
    f"  Notebook result:             {final_notebook_result}",
    "",
    "Claim boundary: this notebook demonstrates deployment/runtime/API behavior and",
    "a deterministic semantic sanity task. It does not claim SLA, HA, multi-tenancy,",
    "mTLS, service mesh, or DFlash2 speedup versus a matched baseline.",
]

review_summary = "\n".join(summary_lines) + "\n"
REVIEW_SUMMARY_PATH.write_text(review_summary, encoding="utf-8")
print("\n=== PUBLICATION REVIEW SUMMARY ===")
print(review_summary.rstrip())
print("=== END PUBLICATION REVIEW SUMMARY ===")

# Bind the final, secret-free publication artifacts with checksums.
publication_files = [EVIDENCE_PATH, INTERACTION_EVIDENCE_PATH, REVIEW_SUMMARY_PATH]
for path in publication_files:
    assert path.is_file(), f"Missing publication artifact: {path}"

# Defense in depth: the live bearer token must not occur in any publication artifact.
for path in publication_files:
    text = path.read_text(encoding="utf-8", errors="replace")
    assert token not in text, f"Bearer credential leaked into {path.name}"

checksum_lines = []
for path in publication_files:
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    checksum_lines.append(f"{digest}  {path.name}")
SHA256SUMS_PATH.write_text("\n".join(checksum_lines) + "\n", encoding="utf-8")

if PUBLICATION_EVIDENCE_ZIP_PATH.exists():
    PUBLICATION_EVIDENCE_ZIP_PATH.unlink()
with zipfile.ZipFile(PUBLICATION_EVIDENCE_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in publication_files + [SHA256SUMS_PATH]:
        zf.write(path, arcname=path.name)

zip_sha256 = hashlib.sha256(PUBLICATION_EVIDENCE_ZIP_PATH.read_bytes()).hexdigest()
PUBLICATION_EVIDENCE_ZIP_SHA256_PATH.write_text(
    f"{zip_sha256}  {PUBLICATION_EVIDENCE_ZIP_PATH.name}\n",
    encoding="utf-8",
)

print(f"PUBLICATION_REVIEW_SUMMARY={REVIEW_SUMMARY_PATH}")
print(f"PUBLICATION_SHA256SUMS={SHA256SUMS_PATH}")
print(f"PUBLICATION_EVIDENCE_ZIP={PUBLICATION_EVIDENCE_ZIP_PATH}")
print(f"PUBLICATION_EVIDENCE_ZIP_SHA256={zip_sha256}")
print(f"PUBLICATION_EVIDENCE_ZIP_SHA256_FILE={PUBLICATION_EVIDENCE_ZIP_SHA256_PATH}")
print("PUBLICATION_SECRET_SCAN=PASS")
print("PUBLICATION_ARTIFACT_BUNDLE=PASS")

print(f"FINAL_CORE_PRODUCTION_DEMO={core_verdict}")
print(f"FINAL_PUBLIC_QUICK_TUNNEL={public_verdict}")
print(f"FINAL_TRANSPORT_BLOCKER={SESSION.get('transport_blocker', 'UNKNOWN')}")
print(f"FINAL_VENV_LOG_POLISH_GATE={SESSION.get('venv_log_polish_gate', 'UNKNOWN')}")
print(f"FINAL_NOTEBOOK_RESULT={final_notebook_result}")
print(f"FINAL_OVERALL_PRODUCTION_DEMO={overall_verdict}")
print(f"FINAL_EVIDENCE_PATH={EVIDENCE_PATH}")
forensic_capture_count = int(SESSION.get("forensic_capture_count", 0))
print(f"FORENSIC_CAPTURE_COUNT={forensic_capture_count}")
if forensic_capture_count > 0 and FORENSIC_ZIP_PATH.exists():
    print(f"FINAL_FORENSIC_ZIP={FORENSIC_ZIP_PATH}")
else:
    print("FINAL_FORENSIC_ZIP=NOT_CREATED_THIS_RUN")

if AUTO_CLEANUP_AT_END:
    assert SESSION.get("public_url_lifecycle") == "HISTORICAL_TEST_ENDPOINT_STOPPED"
    print("PUBLIC_URL_LIFECYCLE=HISTORICAL_TEST_ENDPOINT_STOPPED")

print("RERUN_SAME_NOTEBOOK=SUPPORTED")
print("NOTEBOOK_EXECUTION_COMPLETED=PASS")
